# Classical Model Benchmarking for GBM Radiotherapy scGPT

This refined notebook has two goals: compute benchmark results for the project models, then contextualise the results against published perturbation-prediction benchmarks. The published studies are used as methodological context, not as direct numerical comparators, because they mostly use open-source genetic or drug perturbation datasets rather than GBM radiotherapy.

## Notebook Design

- Use the same metric definitions across sanity baselines, condition-only classical models and the final scGPT output.
- Keep evaluation-source metadata visible, because the scGPT Myriad output is loaded from `test_results.npz`.
- Report-facing metrics are pruned to avoid redundancy: Pearson, Pearson DE, MSE, Pearson Delta, Pearson DE Delta, AUPRC and full-gene distributional distances.
- Spearman and Common-DEGs are not emphasised in the report tables because they overlap with Pearson Delta / Pearson DE Delta and AUPRC respectively.
- Treat condition-mean and condition-only models as baselines, not mechanistic models of single-cell heterogeneity.


## 0. Imports and Configuration

In [ ]:
#!/usr/bin/env python
# coding: utf-8
"""
Refined benchmark analysis for the MSc scGPT GBM radiotherapy project.

Notebook goals:
1. Compute project-level benchmark results for sanity baselines, condition-only
   classical models, and the final scGPT Myriad test output.
2. Produce report-ready metric tables and figures using a focused metric set.
3. Position this GBM radiotherapy dataset against published perturbation
   prediction benchmarks, while making clear that those studies use different
   open-source genetic or drug perturbation datasets.
"""

import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.linalg as la
import seaborn as sns
from scipy.spatial.distance import cdist
from scipy.stats import pearsonr, spearmanr, wasserstein_distance, ttest_ind
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNetCV, MultiTaskElasticNet
from sklearn.neighbors import KNeighborsRegressor, NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_recall_curve, auc

# For Jupyter display
from IPython.display import display

warnings.filterwarnings("ignore")


# ==========================================================
# 0. Configuration
# ==========================================================
save_dir = Path("./dev_perturb_adamson-Jun28-15-05")
output_dir = Path("./refined_baseline_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

target_conditions = ["IR-24h", "IR-1week"]
all_conditions = ["ctrl"] + target_conditions
dummy_genes = ["IR-24h", "IR-1week", "IR_24h", "IR_1week"]
top_k_de = 20
random_state = 42




## 1-2. Data Alignment and Metric Functions

In [ ]:
# ==========================================================
# 1. Loading and alignment helpers
# ==========================================================
def standardize_condition_names(meta: pd.DataFrame) -> pd.DataFrame:
    meta = meta.copy()
    meta["condition"] = meta["condition"].astype(str).str.replace("_", "-", regex=False)
    return meta


def load_saved_split(run_dir: Path):
    x_train = np.load(run_dir / "train_expression.npy")
    x_test = np.load(run_dir / "test_expression.npy")
    meta_train = standardize_condition_names(pd.read_csv(run_dir / "train_metadata.csv"))
    meta_test = standardize_condition_names(pd.read_csv(run_dir / "test_metadata.csv"))

    with open(run_dir / "gene_names.json", "r") as f:
        gene_names_full = np.array(json.load(f))

    bio_mask = ~np.isin(gene_names_full, dummy_genes)
    gene_names = gene_names_full[bio_mask]
    x_train = x_train[:, bio_mask]
    x_test = x_test[:, bio_mask]
    return x_train, x_test, meta_train, meta_test, gene_names, bio_mask, gene_names_full


def align_scgpt_results(run_dir: Path, bio_mask: np.ndarray, gene_names_full: np.ndarray):
    results_path = run_dir / "test_results.npz"
    if not results_path.exists():
        return None

    results = np.load(results_path)
    pred = results["pred"]
    truth = results["truth"]
    pert_cat = np.array([str(x).replace("_", "-") for x in results["pert_cat"]])

    if pred.shape[1] == len(gene_names_full):
        pred = pred[:, bio_mask]
        truth = truth[:, bio_mask]
    elif pred.shape[1] == bio_mask.sum():
        pass
    elif pred.shape[1] == len(gene_names_full) - len(["IR-24h", "IR-1week"]):
        no_hyphen_mask = ~np.isin(gene_names_full, ["IR-24h", "IR-1week"])
        gene_names_no_hyphen = gene_names_full[no_hyphen_mask]
        no_underscore_mask = ~np.isin(gene_names_no_hyphen, ["IR_24h", "IR_1week"])
        pred = pred[:, no_underscore_mask]
        truth = truth[:, no_underscore_mask]
    else:
        raise ValueError(
            f"Unexpected scGPT prediction dimension: {pred.shape[1]} columns, "
            f"but gene_names has {len(gene_names_full)} and biological genes has {bio_mask.sum()}."
        )

    meta = pd.DataFrame({"condition": pert_cat})
    return pred, truth, meta


# ==========================================================
# 2. Metrics (including AUPRC, MSE, Local E‑distance, Spearman)
# ==========================================================
def safe_pearson(x, y) -> float:
    x = np.asarray(x)
    y = np.asarray(y)
    if x.size < 2 or y.size < 2:
        return 0.0
    if np.std(x) < 1e-12 or np.std(y) < 1e-12:
        return 0.0
    return float(pearsonr(x, y)[0])


def safe_spearman(x, y) -> float:
    x = np.asarray(x)
    y = np.asarray(y)
    if x.size < 2 or y.size < 2:
        return 0.0
    if np.std(x) < 1e-12 or np.std(y) < 1e-12:
        return 0.0
    return float(spearmanr(x, y)[0])


def energy_distance_subset(pred, truth, max_cells=400, random_state=42) -> float:
    rng = np.random.default_rng(random_state)
    n_pred = pred.shape[0]
    n_truth = truth.shape[0]
    if n_pred == 0 or n_truth == 0:
        return np.nan
    if n_pred > max_cells:
        pred = pred[rng.choice(n_pred, max_cells, replace=False)]
    if n_truth > max_cells:
        truth = truth[rng.choice(n_truth, max_cells, replace=False)]

    cross = cdist(pred, truth, metric="euclidean").mean()
    within_pred = cdist(pred, pred, metric="euclidean").mean()
    within_truth = cdist(truth, truth, metric="euclidean").mean()
    return float(2 * cross - within_pred - within_truth)


def local_energy_distance_subset(pred, truth, k=30, max_cells=400, random_state=42) -> float:
    """
    Local E‑distance (negative Euclidean kernel) as defined in Heidari et al.
    Only pairs within the k‑nearest neighbours are used.
    """
    rng = np.random.default_rng(random_state)
    n_pred = pred.shape[0]
    n_truth = truth.shape[0]
    if n_pred == 0 or n_truth == 0:
        return np.nan
    if n_pred > max_cells:
        pred = pred[rng.choice(n_pred, max_cells, replace=False)]
        n_pred = pred.shape[0]
    if n_truth > max_cells:
        truth = truth[rng.choice(n_truth, max_cells, replace=False)]
        n_truth = truth.shape[0]

    k = min(k, n_pred + n_truth - 1)  # avoid exceeding total points
    combined = np.vstack([pred, truth])
    nbrs = NearestNeighbors(n_neighbors=k, metric="euclidean").fit(combined)
    distances, indices = nbrs.kneighbors(combined)

    # Split indices into pred and truth groups
    pred_mask = np.arange(n_pred)
    truth_mask = np.arange(n_pred, n_pred + n_truth)

    total = 0.0
    for i in range(n_pred):
        neigh = indices[i]
        within_pred = neigh[np.isin(neigh, pred_mask)]
        if len(within_pred) > 0:
            total += np.mean(-distances[i, np.isin(neigh, pred_mask)])
        cross_pred_to_truth = neigh[np.isin(neigh, truth_mask)]
        if len(cross_pred_to_truth) > 0:
            total -= np.mean(-distances[i, np.isin(neigh, truth_mask)])

    for j in range(n_truth):
        idx = j + n_pred
        neigh = indices[idx]
        within_truth = neigh[np.isin(neigh, truth_mask)]
        if len(within_truth) > 0:
            total += np.mean(-distances[idx, np.isin(neigh, truth_mask)])
        cross_truth_to_pred = neigh[np.isin(neigh, pred_mask)]
        if len(cross_truth_to_pred) > 0:
            total -= np.mean(-distances[idx, np.isin(neigh, pred_mask)])

    total /= (n_pred + n_truth)
    return float(total)


def mean_wasserstein(pred, truth, gene_idx) -> float:
    vals = [
        wasserstein_distance(pred[:, g], truth[:, g])
        for g in gene_idx
        if pred.shape[0] > 0 and truth.shape[0] > 0
    ]
    return float(np.mean(vals)) if vals else np.nan


def mean_kl_divergence(pred, truth, gene_idx, bins=50, eps=1e-8) -> float:
    vals = []
    for g in gene_idx:
        p_g = truth[:, g]
        q_g = pred[:, g]
        lo = min(p_g.min(), q_g.min())
        hi = max(p_g.max(), q_g.max())
        if hi - lo < 1e-12:
            vals.append(0.0)
            continue

        p_counts, _ = np.histogram(p_g, bins=bins, range=(lo, hi))
        q_counts, _ = np.histogram(q_g, bins=bins, range=(lo, hi))
        p = p_counts.astype(float) + eps
        q = q_counts.astype(float) + eps
        p /= p.sum()
        q /= q.sum()
        vals.append(float(np.sum(p * np.log(p / q))))

    return float(np.mean(vals)) if vals else np.nan


def compute_auprc(pred_delta, truth_pvals, threshold=0.05) -> float:
    """
    Use absolute predicted delta as the score and uncorrected Welch-test
    labels from the observed test cells as the positive class.
    """
    if len(truth_pvals) == 0:
        return np.nan
    labels = (truth_pvals < threshold).astype(int)
    scores = np.abs(pred_delta)
    precision, recall, _ = precision_recall_curve(labels, scores)
    return float(auc(recall, precision))


def compute_all_metrics(pred, truth, metadata, ctrl_mean, top_k=20):
    """
    Calculates matched pseudobulk, distributional, and DEG recovery metrics.

    Adds:
    - AUPRC (Welch-test labels and absolute predicted-delta scores)
    - MSE: squared error of the predicted condition-level mean expression
    - Local E‑distance (DE and all genes)
    - Spearman correlations (mean, delta, delta DE)
    """
    results = {}
    metadata = standardize_condition_names(metadata)

    ctrl_mask = metadata["condition"] == "ctrl"
    ctrl_X = truth[ctrl_mask] if np.any(ctrl_mask) and len(truth) == len(metadata) else None

    for cond in target_conditions:
        mask = metadata["condition"].to_numpy() == cond
        if not np.any(mask):
            continue

        pred_cond = pred[mask]
        truth_cond = truth[mask]

        pred_mean = pred_cond.mean(axis=0)
        truth_mean = truth_cond.mean(axis=0)
        pred_delta = pred_mean - ctrl_mean
        truth_delta = truth_mean - ctrl_mean
        mse = float(np.mean((pred_mean - truth_mean) ** 2))

        all_idx = np.arange(pred_cond.shape[1])
        top_idx = np.argsort(np.abs(truth_delta))[-top_k:]
        pred_top_idx = np.argsort(np.abs(pred_delta))[-top_k:]
        common_de = len(set(top_idx) & set(pred_top_idx)) / top_k

        # ---- AUPRC: observed Welch-test labels and absolute prediction scores ----
        auprc_val = np.nan
        if ctrl_X is not None:
            pvals = np.asarray([
                ttest_ind(truth_cond[:, g], ctrl_X[:, g], equal_var=False).pvalue
                for g in range(truth_cond.shape[1])
            ])
            auprc_val = compute_auprc(pred_delta, pvals)

        # ---- Local E‑distance ----
        k_neighbors = min(30, pred_cond.shape[0] - 1, truth_cond.shape[0] - 1)
        if k_neighbors >= 2:
            local_energy_de = local_energy_distance_subset(pred_cond[:, top_idx], truth_cond[:, top_idx], k=k_neighbors)
            local_energy_all = local_energy_distance_subset(pred_cond, truth_cond, k=k_neighbors)
        else:
            local_energy_de = np.nan
            local_energy_all = np.nan

        # ---- Spearman metrics ----
        spearman_val = safe_spearman(pred_mean, truth_mean)
        spearman_delta = safe_spearman(pred_delta, truth_delta)
        spearman_delta_de = safe_spearman(pred_delta[top_idx], truth_delta[top_idx])

        results[cond] = {
            "pearson": safe_pearson(pred_mean, truth_mean),
            "pearson_de": safe_pearson(pred_mean[top_idx], truth_mean[top_idx]),
            "pearson_delta": safe_pearson(pred_delta, truth_delta),
            "pearson_delta_de": safe_pearson(pred_delta[top_idx], truth_delta[top_idx]),
            "mse": mse,
            "spearman": spearman_val,
            "spearman_delta": spearman_delta,
            "spearman_delta_de": spearman_delta_de,
            "auprc": auprc_val,
            "wasserstein_de": mean_wasserstein(pred_cond, truth_cond, top_idx),
            "energy_de": energy_distance_subset(pred_cond[:, top_idx], truth_cond[:, top_idx]),
            "local_energy_de": local_energy_de,
            "kl_de": mean_kl_divergence(pred_cond, truth_cond, top_idx),
            "wasserstein_all": mean_wasserstein(pred_cond, truth_cond, all_idx),
            "energy_all": energy_distance_subset(pred_cond, truth_cond),
            "local_energy_all": local_energy_all,
            "kl_all": mean_kl_divergence(pred_cond, truth_cond, all_idx),
            "common_degs": float(common_de),
        }

    # Average over conditions
    avg = {}
    if results:
        for metric in next(iter(results.values())).keys():
            vals = [results[c][metric] for c in results if not np.isnan(results[c][metric])]
            avg[metric] = float(np.mean(vals)) if vals else np.nan
    results["average"] = avg
    return results


def flatten_results(results_by_model):
    rows = []
    for model_name, result in results_by_model.items():
        for cond in target_conditions + ["average"]:
            if cond not in result:
                continue
            row = {"model": model_name, "condition": cond}
            row.update(result[cond])
            rows.append(row)
    return pd.DataFrame(rows)


def add_minmax_normalized_distance_metrics(results_by_model):
    """
    Adds Wei-style min-max normalized values for unbounded distance metrics.
    """
    distance_metrics = [
        "wasserstein_de", "energy_de", "local_energy_de", "kl_de",
        "wasserstein_all", "energy_all", "local_energy_all", "kl_all",
    ]

    for cond in target_conditions:
        for metric in distance_metrics:
            values = []
            model_names = []
            for model_name, model_result in results_by_model.items():
                if cond not in model_result:
                    continue
                value = model_result[cond].get(metric, np.nan)
                if np.isfinite(value):
                    values.append(value)
                    model_names.append(model_name)

            if not values:
                continue

            values = np.asarray(values, dtype=float)
            min_value = values.min()
            max_value = values.max()
            denom = max_value - min_value
            norm_name = f"{metric}_norm"

            for model_name, value in zip(model_names, values):
                if denom < 1e-12:
                    results_by_model[model_name][cond][norm_name] = 0.0
                else:
                    results_by_model[model_name][cond][norm_name] = float((value - min_value) / denom)

    # Recompute averages after adding normalized scores.
    for model_result in results_by_model.values():
        avg = {}
        keys = sorted({k for cond in target_conditions if cond in model_result for k in model_result[cond].keys()})
        for key in keys:
            vals = [model_result[cond][key] for cond in target_conditions if cond in model_result]
            avg[key] = float(np.nanmean(vals))
        model_result["average"] = avg


def print_metric_table(summary_df: pd.DataFrame) -> None:
    """
    Display metric tables using IPython.display for Jupyter notebooks.
    """
    metric_cols = [
        "pearson", "pearson_de", "mse", "pearson_delta", "pearson_delta_de",
        "spearman", "spearman_delta", "spearman_delta_de",
        "auprc",
        "common_degs",
        "wasserstein_de", "energy_de", "local_energy_de", "kl_de",
        "wasserstein_all", "energy_all", "local_energy_all", "kl_all",
    ]
    metric_labels = {
        "pearson": "Pearson",
        "pearson_de": "Pearson DE",
        "pearson_delta": "Pearson Delta",
        "pearson_delta_de": "Pearson DE Delta",
        "mse": "MSE",
        "spearman": "Spearman",
        "spearman_delta": "Spearman Delta",
        "spearman_delta_de": "Spearman DE Delta",
        "auprc": "AUPRC",
        "common_degs": "Common DEGs",
        "wasserstein_de": "Wasserstein DE",
        "energy_de": "Energy DE",
        "local_energy_de": "Local E DE",
        "kl_de": "KL DE",
        "wasserstein_all": "Wasserstein All",
        "energy_all": "Energy All",
        "local_energy_all": "Local E All",
        "kl_all": "KL All",
    }

    # Main table with raw distances
    table = summary_df[["model", "condition"] + metric_cols].copy()
    table = table.rename(columns=metric_labels)
    print("\nFocused comparison table with raw distances:")
    display(table)

    # Average across timepoints
    avg_table = table[table["condition"] == "average"].copy()
    avg_table = avg_table.sort_values("Local E DE", ascending=True)
    print("\nAverage across RT timepoints (sorted by Local E DE):")
    display(avg_table)

    # Normalized distance metrics
    norm_cols = [
        "wasserstein_de_norm", "energy_de_norm", "local_energy_de_norm", "kl_de_norm",
        "wasserstein_all_norm", "energy_all_norm", "local_energy_all_norm", "kl_all_norm",
    ]
    norm_labels = {
        "wasserstein_de_norm": "Wasserstein DE Norm",
        "energy_de_norm": "Energy DE Norm",
        "local_energy_de_norm": "Local E DE Norm",
        "kl_de_norm": "KL DE Norm",
        "wasserstein_all_norm": "Wasserstein All Norm",
        "energy_all_norm": "Energy All Norm",
        "local_energy_all_norm": "Local E All Norm",
        "kl_all_norm": "KL All Norm",
    }
    norm_table = summary_df[["model", "condition"] + norm_cols].copy().rename(columns=norm_labels)
    print("\nWei-style min-max normalized distance metrics (lower is better):")
    display(norm_table)

    avg_norm_table = norm_table[norm_table["condition"] == "average"].copy()
    avg_norm_table = avg_norm_table.sort_values("Local E DE Norm", ascending=True)
    print("\nNormalized average distance metrics (lower is better):")
    display(avg_norm_table)



## 3-4. Prediction Helpers and Data Loading

In [ ]:
# ==========================================================
# 3. Prediction helpers
# ==========================================================
def predict_from_condition_vectors(meta, condition_to_vector, fallback):
    pred = np.zeros((len(meta), len(fallback)), dtype=np.float32)
    for i, (_, row) in enumerate(meta.iterrows()):
        pred[i] = condition_to_vector.get(row["condition"], fallback)
    return pred


def build_condition_features(meta, cond_to_encoding):
    return np.asarray([cond_to_encoding[row["condition"]] for _, row in meta.iterrows()])


def add_ctrl_mean(delta_pred, ctrl_mean):
    return delta_pred + ctrl_mean.reshape(1, -1)


# ==========================================================
# 4. Load data
# ==========================================================
X_train, X_test, meta_train, meta_test, gene_names, bio_mask, gene_names_full = load_saved_split(save_dir)

ctrl_mask_train = meta_train["condition"] == "ctrl"
if not np.any(ctrl_mask_train):
    raise ValueError("No control cells found in the training split.")

ctrl_mean = X_train[ctrl_mask_train].mean(axis=0)
Y_train_delta = X_train - ctrl_mean.reshape(1, -1)

print(f"Loaded data from: {save_dir}")
print(f"Train cells: {X_train.shape[0]} | Test cells: {X_test.shape[0]} | Biological genes: {X_train.shape[1]}")
print("Train condition counts:")
print(meta_train["condition"].value_counts())
print("Test condition counts:")
print(meta_test["condition"].value_counts())
print(f"Using top_k_de = {top_k_de}")




## 5-7. Benchmark Models and scGPT Evaluation

In [ ]:
# ==========================================================
# 5. Sanity baselines
# ==========================================================
results_by_model = {}

# No-effect baseline
pred_control = np.tile(ctrl_mean, (X_test.shape[0], 1))
results_by_model["Control mean (no RT effect)"] = compute_all_metrics(
    pred_control, X_test, meta_test, ctrl_mean, top_k=top_k_de
)

# Seen-condition mean
cond_means = {}
for cond in all_conditions:
    mask = meta_train["condition"] == cond
    cond_means[cond] = X_train[mask].mean(axis=0) if np.any(mask) else ctrl_mean

pred_cond_mean = predict_from_condition_vectors(meta_test, cond_means, fallback=ctrl_mean)
results_by_model["Seen-condition mean"] = compute_all_metrics(
    pred_cond_mean, X_test, meta_test, ctrl_mean, top_k=top_k_de
)

# Seen-condition delta mean
cond_delta_means = {cond: cond_means[cond] - ctrl_mean for cond in all_conditions}
pred_delta_mean = add_ctrl_mean(
    predict_from_condition_vectors(meta_test, cond_delta_means, fallback=np.zeros_like(ctrl_mean)),
    ctrl_mean,
)
results_by_model["Seen-condition delta mean"] = compute_all_metrics(
    pred_delta_mean, X_test, meta_test, ctrl_mean, top_k=top_k_de
)


# ==========================================================
# 6. Delta-target classical models
# ==========================================================
cond_to_onehot = {
    "ctrl": np.array([1.0, 0.0, 0.0]),
    "IR-24h": np.array([0.0, 1.0, 0.0]),
    "IR-1week": np.array([0.0, 0.0, 1.0]),
}

X_train_cond = build_condition_features(meta_train, cond_to_onehot)
X_test_cond = build_condition_features(meta_test, cond_to_onehot)

# 6.1 Multi-task Elastic Net
enet_direct = MultiTaskElasticNet(alpha=0.1, l1_ratio=0.5, random_state=random_state, max_iter=1000)
enet_direct.fit(X_train_cond, Y_train_delta)
pred_enet_delta = add_ctrl_mean(enet_direct.predict(X_test_cond), ctrl_mean)
results_by_model["Elastic Net delta"] = compute_all_metrics(
    pred_enet_delta, X_test, meta_test, ctrl_mean, top_k=top_k_de
)

# 6.2 Bilinear ridge
n_gene_components = min(20, X_train.shape[1], X_train.shape[0] - 1)
ridge_penalty = 0.1

X_train_centered = X_train - X_train.mean(axis=0, keepdims=True)
pca_gene = PCA(n_components=n_gene_components, random_state=random_state)
A = pca_gene.fit_transform(X_train_centered.T)

delta_pseudobulk = np.column_stack([cond_delta_means[c] for c in all_conditions])
B = np.eye(len(all_conditions), dtype=float)

ATA = A.T @ A
ATA_reg = ATA + ridge_penalty * np.eye(ATA.shape[0])
BBT = B @ B.T
BBT_reg = BBT + ridge_penalty * np.eye(BBT.shape[0])
K = la.solve(ATA_reg, A.T @ delta_pseudobulk @ B.T @ la.inv(BBT_reg))

pred_delta_pseudobulk = A @ K @ B
cond_to_bilinear_delta = {cond: pred_delta_pseudobulk[:, i] for i, cond in enumerate(all_conditions)}
pred_bilinear = add_ctrl_mean(
    predict_from_condition_vectors(meta_test, cond_to_bilinear_delta, fallback=np.zeros_like(ctrl_mean)),
    ctrl_mean,
)
results_by_model["Bilinear Ridge delta"] = compute_all_metrics(
    pred_bilinear, X_test, meta_test, ctrl_mean, top_k=top_k_de
)

# 6.3 Elastic Net CV in PCA space
enet_cv = ElasticNetCV(
    cv=3,
    l1_ratio=[0.1, 0.5, 0.9],
    alphas=[0.01, 0.1, 1.0],
    random_state=random_state,
    max_iter=2000,
)
K_list = []
for i, cond in enumerate(all_conditions):
    enet_cv.fit(A, delta_pseudobulk[:, i])
    K_list.append(enet_cv.coef_)
    print(f"[{cond}] ElasticNetCV alpha={enet_cv.alpha_}, l1_ratio={enet_cv.l1_ratio_}")

K_enet = np.column_stack(K_list)
pred_delta_pseudobulk_enet = A @ K_enet
cond_to_enet_pca_delta = {cond: pred_delta_pseudobulk_enet[:, i] for i, cond in enumerate(all_conditions)}
pred_enet_pca = add_ctrl_mean(
    predict_from_condition_vectors(meta_test, cond_to_enet_pca_delta, fallback=np.zeros_like(ctrl_mean)),
    ctrl_mean,
)
results_by_model["Elastic Net PCA delta"] = compute_all_metrics(
    pred_enet_pca, X_test, meta_test, ctrl_mean, top_k=top_k_de
)

# 6.4 Pseudobulk RF-PCA
# Aggregate by sample and malignant cell type, predict PCA-compressed deltas,
# then map each pseudobulk prediction back to its held-out test cells.
from types import SimpleNamespace
from pseudobulk_rf_pca import (
    GROUP_COLUMNS as RF_PCA_GROUP_COLUMNS,
    add_control_reference as rf_pca_add_control_reference,
    aggregate_pseudobulk as rf_pca_aggregate,
    build_control_references as rf_pca_control_references,
    fit_predict_rf_pca,
    subtract_control_reference as rf_pca_subtract_control_reference,
)

rf_pca_train_groups, rf_pca_train_expression = rf_pca_aggregate(X_train, meta_train)
rf_pca_test_groups, rf_pca_test_expression = rf_pca_aggregate(X_test, meta_test)
rf_pca_references = rf_pca_control_references(
    rf_pca_train_groups, rf_pca_train_expression
)
rf_pca_train_delta = rf_pca_subtract_control_reference(
    rf_pca_train_groups, rf_pca_train_expression, rf_pca_references
)
rf_pca_args = SimpleNamespace(
    n_components=10,
    n_estimators=500,
    max_depth=6,
    min_samples_leaf=2,
    max_features=0.7,
    random_state=random_state,
)
rf_pca_test_delta, rf_pca_importance, rf_pca_diagnostics = fit_predict_rf_pca(
    rf_pca_train_groups,
    rf_pca_train_delta,
    rf_pca_test_groups,
    rf_pca_args,
)
rf_pca_test_group_predictions = rf_pca_add_control_reference(
    rf_pca_test_groups, rf_pca_test_delta, rf_pca_references
)

rf_pca_lookup = {
    tuple(row[column] for column in RF_PCA_GROUP_COLUMNS): prediction
    for (_, row), prediction in zip(
        rf_pca_test_groups.iterrows(), rf_pca_test_group_predictions
    )
}
pred_rf_pca = np.vstack([
    rf_pca_lookup[tuple(row[column] for column in RF_PCA_GROUP_COLUMNS)]
    for _, row in meta_test.iterrows()
]).astype(np.float32)

results_by_model["Pseudobulk RF-PCA"] = compute_all_metrics(
    pred_rf_pca, X_test, meta_test, ctrl_mean, top_k=top_k_de
)
rf_pca_importance.sort_values("importance", ascending=False).to_csv(
    output_dir / "rf_pca_feature_importance_saved_split.csv", index=False
)
pd.DataFrame([rf_pca_diagnostics]).to_csv(
    output_dir / "rf_pca_diagnostics_saved_split.csv", index=False
)
print(
    "Pseudobulk RF-PCA: "
    f"{len(rf_pca_train_groups)} training pseudobulks, "
    f"{rf_pca_diagnostics['n_components']} delta PCs, "
    f"{rf_pca_diagnostics['explained_variance']:.1%} explained variance."
)

# 6.5 KNN
scaler = StandardScaler()
X_train_cond_scaled = scaler.fit_transform(X_train_cond)
X_test_cond_scaled = scaler.transform(X_test_cond)

knn = KNeighborsRegressor(n_neighbors=5, n_jobs=-1)
knn.fit(X_train_cond_scaled, Y_train_delta)
pred_knn = add_ctrl_mean(knn.predict(X_test_cond_scaled), ctrl_mean)
results_by_model["KNN delta"] = compute_all_metrics(
    pred_knn, X_test, meta_test, ctrl_mean, top_k=top_k_de
)


# ==========================================================
# 7. scGPT, evaluated with the exact same function
# ==========================================================
scgpt_payload = align_scgpt_results(save_dir, bio_mask=bio_mask, gene_names_full=gene_names_full)
if scgpt_payload is not None:
    scgpt_pred, scgpt_truth, scgpt_meta = scgpt_payload
    results_by_model["scGPT"] = compute_all_metrics(
        scgpt_pred, scgpt_truth, scgpt_meta, ctrl_mean, top_k=top_k_de
    )
    print(
        f"scGPT Myriad test output: {scgpt_truth.shape[0]} rows, "
        f"{scgpt_truth.shape[1]} biological genes after dummy-gene removal."
    )
    if scgpt_truth.shape[0] != X_test.shape[0]:
        print(
            "NOTE: scGPT is evaluated from test_results.npz, whereas the classical "
            "baselines below use the local saved test_expression.npy split. Treat the "
            "comparison as project-level benchmarking with shared metric definitions, "
            "not as a paired row-by-row comparison."
        )
else:
    print("No test_results.npz found, so scGPT metrics were not added.")

model_context = {}
for model_name in results_by_model:
    if model_name == "Control mean (no RT effect)":
        role = "sanity baseline"
    elif model_name in {"Seen-condition mean", "Seen-condition delta mean"}:
        role = "condition mean sanity baseline"
    elif model_name == "scGPT":
        role = "scGPT final model"
    elif model_name == "Pseudobulk RF-PCA":
        role = "sample x cell-type pseudobulk RF-PCA baseline"
    else:
        role = "condition-only classical baseline"
    model_context[model_name] = {
        "model_role": role,
        "evaluation_source": "local saved split: test_expression.npy",
        "n_eval_cells": int(X_test.shape[0]),
    }

if scgpt_payload is not None:
    model_context["scGPT"] = {
        "model_role": "scGPT final model",
        "evaluation_source": "Myriad final test_results.npz",
        "n_eval_cells": int(scgpt_truth.shape[0]),
    }

add_minmax_normalized_distance_metrics(results_by_model)




## 8. Report-Focused Benchmark Outputs

In [ ]:
# ==========================================================
# 8. Save report-ready tables and plots
# ==========================================================
with open(output_dir / "refined_model_metrics.json", "w") as f:
    json.dump(results_by_model, f, indent=2)

summary_df = flatten_results(results_by_model)
summary_df["model_role"] = summary_df["model"].map(lambda m: model_context.get(m, {}).get("model_role", "unknown"))
summary_df["evaluation_source"] = summary_df["model"].map(lambda m: model_context.get(m, {}).get("evaluation_source", "unknown"))
summary_df["n_eval_cells"] = summary_df["model"].map(lambda m: model_context.get(m, {}).get("n_eval_cells", np.nan))
summary_df.to_csv(output_dir / "refined_model_metrics.csv", index=False)

# Full metric table is retained for transparency, but the report-facing table
# focuses on metrics that answer the thesis question most clearly.
# Spearman and Common-DEGs are deliberately excluded here because they mostly
# repeat the interpretation of Pearson-family delta metrics and AUPRC.
report_metrics = [
    "pearson",                # global expression-profile reconstruction
    "pearson_de",             # reconstruction over strongest observed perturbation genes
    "pearson_delta",          # perturbation-shift mean agreement
    "pearson_delta_de",       # perturbation-shift agreement on strongest observed genes
    "mse",                    # squared error of the predicted condition-level mean expression
    "auprc",                  # DEG recovery from predicted delta magnitude
    "energy_all",             # full-distribution discrepancy
    "local_energy_all",       # local full-distribution discrepancy over all genes
    "wasserstein_all",        # marginal full-gene transport distance
    "kl_all",                 # marginal distribution divergence
]
metric_labels = {
    "pearson": "Pearson",
    "pearson_de": "Pearson DE",
    "pearson_delta": "Pearson Delta",
    "pearson_delta_de": "Pearson DE Delta",
    "mse": "MSE",
    "auprc": "AUPRC",
    "energy_all": "E-distance All",
    "local_energy_all": "Local E-distance All",
    "wasserstein_all": "Wasserstein All",
    "kl_all": "KL All",
}
higher_is_better = {
    "pearson": True,
    "pearson_de": True,
    "pearson_delta": True,
    "pearson_delta_de": True,
    "mse": False,
    "auprc": True,
    "energy_all": False,
    "local_energy_all": False,
    "wasserstein_all": False,
    "kl_all": False,
}

report_cols = ["model", "model_role", "evaluation_source", "n_eval_cells", "condition"] + report_metrics
report_df = summary_df[report_cols].copy()
report_df.to_csv(output_dir / "report_focused_model_metrics.csv", index=False)

avg_report_df = report_df[report_df["condition"] == "average"].copy()
avg_report_df.to_csv(output_dir / "report_focused_average_metrics.csv", index=False)

print("\nReport-focused metrics:")
display(report_df.rename(columns=metric_labels))

print("\nAverage across IR-24h and IR-1week:")
display(avg_report_df.rename(columns=metric_labels))

# Rank table: useful for writing the Results section without over-reading one metric.
rank_rows = []
for condition in target_conditions + ["average"]:
    df_c = summary_df[summary_df["condition"] == condition].copy()
    for metric in report_metrics:
        metric_df = df_c[["model", metric]].dropna().copy()
        if metric_df.empty:
            continue
        ascending = not higher_is_better[metric]
        metric_df["rank"] = metric_df[metric].rank(ascending=ascending, method="min").astype(int)
        metric_df = metric_df.sort_values(metric, ascending=ascending)
        best = metric_df.iloc[0]
        scgpt_match = metric_df[metric_df["model"] == "scGPT"]
        rank_rows.append({
            "condition": condition,
            "metric": metric,
            "metric_label": metric_labels[metric],
            "higher_is_better": higher_is_better[metric],
            "best_model": best["model"],
            "best_value": float(best[metric]),
            "scgpt_rank": int(scgpt_match["rank"].iloc[0]) if not scgpt_match.empty else np.nan,
            "scgpt_value": float(scgpt_match[metric].iloc[0]) if not scgpt_match.empty else np.nan,
            "n_models": int(len(metric_df)),
        })

rank_summary_df = pd.DataFrame(rank_rows)
rank_summary_df.to_csv(output_dir / "report_metric_rank_summary.csv", index=False)
print("\nMetric-level rank summary:")
display(rank_summary_df)

# Compact report figures. DE-local energy is intentionally excluded; the local
# neighbourhood metric is more interpretable over all biological genes.
selected_plot_metrics = [
    "pearson",
    "pearson_delta",
    "pearson_delta_de",
    "mse",
    "auprc",
    "local_energy_all",
    "energy_all",
    "wasserstein_all",
    "kl_all",
]

def plot_lollipop_metrics(df, condition, metrics, output_path):
    df_c = df[df["condition"] == condition].copy()
    n_metrics = len(metrics)
    cols = 3
    rows = int(np.ceil(n_metrics / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 6.2, rows * 4.2))
    axes = np.ravel(axes)
    for i, metric in enumerate(metrics):
        ax = axes[i]
        data = df_c[["model", metric]].dropna().copy()
        ascending = not higher_is_better[metric]
        data = data.sort_values(metric, ascending=ascending)
        y_pos = np.arange(len(data))
        ax.scatter(data[metric], y_pos, color="#1f4e79", s=58, zorder=3)
        xmin = min(0, data[metric].min())
        ax.hlines(y=y_pos, xmin=xmin, xmax=data[metric], color="#9aa0a6", linewidth=1.1, alpha=0.8)
        ax.set_yticks(y_pos)
        ax.set_yticklabels(data["model"])
        ax.invert_yaxis()
        ax.set_xlabel("Score")
        direction = "higher is better" if higher_is_better[metric] else "lower is better"
        ax.set_title(f"{metric_labels[metric]}\n({direction})")
        ax.grid(axis="x", linestyle="--", alpha=0.25)
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
    fig.suptitle(f"Benchmark metrics: {condition}", fontsize=14, fontweight="bold")
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

for condition in target_conditions + ["average"]:
    plot_lollipop_metrics(
        summary_df,
        condition,
        selected_plot_metrics,
        output_dir / f"report_lollipop_metrics_{condition}.png",
    )


def plot_a4_lollipop_all_conditions(df, metrics, output_prefix):
    """
    A4 landscape report figure: each metric panel shows IR-24h, IR-1week,
    and the two-timepoint average together for the same model ordering.
    """
    condition_order = target_conditions + ["average"]
    condition_labels = {"IR-24h": "IR-24h", "IR-1week": "IR-1week", "average": "Average"}
    condition_colors = {"IR-24h": "#2D5A87", "IR-1week": "#C47A36", "average": "#3F8C5A"}
    condition_offsets = {"IR-24h": -0.16, "IR-1week": 0.0, "average": 0.16}

    order_source = df[df["condition"] == "average"][["model", "pearson_delta"]].dropna().copy()
    order_source = order_source.sort_values("pearson_delta", ascending=False)
    model_order = order_source["model"].tolist()
    short_model_labels = {
        "Control mean (no RT effect)": "Control mean",
        "Seen-condition mean": "Seen-cond mean",
        "Seen-condition delta mean": "Seen-cond delta mean",
        "Elastic Net delta": "Elastic Net",
        "Bilinear Ridge delta": "Bilinear Ridge",
        "Elastic Net PCA delta": "Elastic Net PCA",
        "Pseudobulk RF-PCA": "RF-PCA",
        "KNN delta": "KNN",
        "scGPT": "scGPT",
    }

    fig, axes = plt.subplots(3, 3, figsize=(11.69, 8.27), sharey=True)
    axes = np.ravel(axes)
    y_pos = np.arange(len(model_order))

    for i, metric in enumerate(metrics):
        ax = axes[i]
        metric_values = df[df["condition"].isin(condition_order)][metric].replace([np.inf, -np.inf], np.nan).dropna()
        if metric_values.empty:
            ax.axis("off")
            continue

        for y, model in zip(y_pos, model_order):
            model_rows = df[(df["model"] == model) & (df["condition"].isin(condition_order))]
            vals = model_rows[metric].replace([np.inf, -np.inf], np.nan).dropna()
            if not vals.empty:
                ax.hlines(y, vals.min(), vals.max(), color="#B7C0CA", linewidth=0.85, alpha=0.72, zorder=1)
            for cond in condition_order:
                row = model_rows[model_rows["condition"] == cond]
                if row.empty or not np.isfinite(row[metric].iloc[0]):
                    continue
                ax.scatter(
                    row[metric].iloc[0],
                    y + condition_offsets[cond],
                    s=22 if cond != "average" else 30,
                    color=condition_colors[cond],
                    edgecolor="white",
                    linewidth=0.45,
                    zorder=3,
                )

        direction = "higher is better" if higher_is_better[metric] else "lower is better"
        ax.set_title(f"{metric_labels[metric]}\n({direction})", fontsize=7.8, fontweight="bold", pad=4)
        ax.grid(axis="x", linestyle="--", alpha=0.24, linewidth=0.6)
        ax.tick_params(axis="x", labelsize=6.6)
        ax.tick_params(axis="y", length=0, labelsize=6.5)
        ax.spines[["top", "right", "left"]].set_visible(False)

        finite_min = float(metric_values.min())
        finite_max = float(metric_values.max())
        span = max(finite_max - finite_min, 1e-9)
        pad = span * 0.08
        if higher_is_better[metric] and metric != "pearson_delta":
            ax.set_xlim(max(0, finite_min - pad), min(1.05, finite_max + pad))
        elif metric == "pearson_delta":
            ax.set_xlim(min(-0.1, finite_min - pad), min(1.05, finite_max + pad))
        else:
            ax.set_xlim(max(0, finite_min - pad), finite_max + pad)

        if i % 3 == 0:
            ax.set_yticks(y_pos)
            ax.set_yticklabels([short_model_labels.get(m, m) for m in model_order])
        else:
            ax.tick_params(axis="y", labelleft=False)

    for ax in axes[len(metrics):]:
        ax.axis("off")

    axes[0].invert_yaxis()
    legend_handles = [
        plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=condition_colors[c], markeredgecolor="white", markersize=6.2, label=condition_labels[c])
        for c in condition_order
    ]
    fig.legend(legend_handles, [h.get_label() for h in legend_handles], loc="upper center", ncol=3, frameon=False, fontsize=8.2, bbox_to_anchor=(0.58, 0.925))
    fig.suptitle("A4 Summary of Benchmark Metrics Across Radiotherapy Timepoints", fontsize=12.5, fontweight="bold", x=0.52, y=0.975)
    fig.text(
        0.21,
        0.045,
        "Each panel uses the same model order, sorted by average Pearson Delta. Horizontal grey lines show the range across IR-24h, IR-1week and the average.",
        fontsize=6.8,
        color="#4C5663",
    )
    fig.subplots_adjust(left=0.21, right=0.985, top=0.86, bottom=0.11, wspace=0.28, hspace=0.56)

    png_path = output_prefix.with_suffix(".png")
    pdf_path = output_prefix.with_suffix(".pdf")
    svg_path = output_prefix.with_suffix(".svg")
    fig.savefig(png_path, dpi=300)
    fig.savefig(pdf_path)
    fig.savefig(svg_path)
    plt.show()
    return png_path, pdf_path, svg_path


a4_png, a4_pdf, a4_svg = plot_a4_lollipop_all_conditions(
    summary_df,
    selected_plot_metrics,
    output_dir / "report_lollipop_metrics_a4_all_conditions",
)
print(f"Saved A4 combined lollipop figure PNG: {a4_png}")
print(f"Saved A4 combined lollipop figure PDF: {a4_pdf}")
print(f"Saved A4 combined lollipop figure SVG: {a4_svg}")


# A4 portrait contact-sheet figure: the three 3x3 lollipop summaries stacked
# one after another for report placement.
def crop_white_border(image, threshold=248, padding=18):
    arr = np.asarray(image.convert("RGB"))
    non_white = np.any(arr < threshold, axis=2)
    if not np.any(non_white):
        return image
    y_idx, x_idx = np.where(non_white)
    left = max(int(x_idx.min()) - padding, 0)
    right = min(int(x_idx.max()) + padding, image.size[0])
    top = max(int(y_idx.min()) - padding, 0)
    bottom = min(int(y_idx.max()) + padding, image.size[1])
    return image.crop((left, top, right, bottom))


def plot_a4_portrait_stacked_lollipops(output_dir):
    from PIL import Image

    panels = [
        ("A", "IR-24h", output_dir / "report_lollipop_metrics_IR-24h.png"),
        ("B", "IR-1week", output_dir / "report_lollipop_metrics_IR-1week.png"),
        ("C", "Average", output_dir / "report_lollipop_metrics_average.png"),
    ]

    fig, axes = plt.subplots(3, 1, figsize=(8.27, 11.69))
    for ax, (letter, label, path) in zip(axes, panels):
        image = crop_white_border(Image.open(path))
        ax.imshow(image)
        ax.axis("off")
        ax.text(
            0.01,
            0.985,
            f"{letter}. {label}",
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=10.5,
            fontweight="bold",
            color="#111827",
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.82, "pad": 2.5},
        )

    fig.suptitle("Benchmark Metrics Across Radiotherapy Timepoints", fontsize=13.2, fontweight="bold", y=0.982)
    fig.text(
        0.5,
        0.018,
        "A4 portrait contact sheet. Each panel is the full 3x3 lollipop summary for one condition/timepoint.",
        ha="center",
        fontsize=7.2,
        color="#4C5663",
    )
    fig.subplots_adjust(left=0.025, right=0.975, top=0.945, bottom=0.04, hspace=0.055)

    png_path = output_dir / "report_lollipop_metrics_a4_portrait_stacked.png"
    pdf_path = output_dir / "report_lollipop_metrics_a4_portrait_stacked.pdf"
    svg_path = output_dir / "report_lollipop_metrics_a4_portrait_stacked.svg"
    fig.savefig(png_path, dpi=300)
    fig.savefig(pdf_path)
    fig.savefig(svg_path)
    plt.show()
    return png_path, pdf_path, svg_path


portrait_png, portrait_pdf, portrait_svg = plot_a4_portrait_stacked_lollipops(output_dir)
print(f"Saved A4 portrait stacked lollipop PNG: {portrait_png}")
print(f"Saved A4 portrait stacked lollipop PDF: {portrait_pdf}")
print(f"Saved A4 portrait stacked lollipop SVG: {portrait_svg}")


# Standardized relative performance view. This complements the raw lollipop
# plots by putting all metrics on a direction-aware 0-1 scale.
def compute_standardized_performance_scores(df, metrics):
    score_frames = []
    for condition in target_conditions + ["average"]:
        df_c = df[df["condition"] == condition].copy()
        score_df = df_c[["model"]].copy()
        for metric in metrics:
            values = df_c[metric].astype(float)
            finite = values.replace([np.inf, -np.inf], np.nan).dropna()
            if finite.empty:
                score_df[metric] = np.nan
                continue
            if higher_is_better[metric]:
                best = finite.max()
                worst = finite.min()
                denom = best - worst
                if abs(denom) < 1e-12:
                    score_df[metric] = 0.5
                else:
                    score_df[metric] = (values - worst) / denom
            else:
                best = finite.min()
                worst = finite.max()
                denom = worst - best
                if abs(denom) < 1e-12:
                    score_df[metric] = 0.5
                else:
                    score_df[metric] = (worst - values) / denom
        score_df.insert(0, "condition", condition)
        score_frames.append(score_df)
    return pd.concat(score_frames, ignore_index=True)


standardized_scores = compute_standardized_performance_scores(summary_df, selected_plot_metrics)
standardized_scores.to_csv(output_dir / "report_standardized_performance_scores.csv", index=False)

# Keep the raw score tables complete, but hide selected low-priority/duplicative
# models from the report heatmaps for readability.
excluded_heatmap_models = {"KNN delta", "Elastic Net delta", "Elastic Net PCA delta", "Seen-condition mean"}

# Heatmaps omit E-distance All, KL All and Wasserstein All to keep the
# report panel focused on correlation, error, AUPRC and local multivariate structure.
heatmap_plot_metrics = [
    metric
    for metric in selected_plot_metrics
    if metric not in {"energy_all", "kl_all", "wasserstein_all"}
]

# Sort models by the average standardized score in the average-condition panel.
average_scores = standardized_scores[standardized_scores["condition"] == "average"].copy()
average_scores["overall_standardized_score"] = average_scores[heatmap_plot_metrics].mean(axis=1)
standardized_model_order = [
    model
    for model in average_scores.sort_values("overall_standardized_score", ascending=False)["model"].tolist()
    if model not in excluded_heatmap_models
]

short_metric_labels = {
    "pearson": "Pearson",
    "pearson_delta": "Pearson\nDelta",
    "pearson_delta_de": "Pearson\nDE Delta",
    "mse": "MSE",
    "auprc": "AUPRC",
    "local_energy_all": "Local Energy-\ndistance",
    "energy_all": "E-distance\nAll",
    "wasserstein_all": "Wasserstein\nAll",
    "kl_all": "KL\nAll",
}
short_model_labels = {
    "Control mean (no RT effect)": "Control mean",
    "Seen-condition mean": "Seen-cond mean",
    "Seen-condition delta mean": "Seen-cond delta mean",
    "Elastic Net delta": "Elastic Net",
    "Bilinear Ridge delta": "Bilinear Ridge",
    "Elastic Net PCA delta": "Elastic Net PCA",
    "Pseudobulk RF-PCA": "RF-PCA",
    "KNN delta": "KNN",
    "scGPT": "scGPT",
}


def standardized_heatmap_matrix(score_df, condition, model_order, metrics):
    matrix = score_df[score_df["condition"] == condition].set_index("model").reindex(model_order)[metrics]
    return matrix.astype(float)


def add_standardized_heatmap(ax, matrix, title, annotate=False, show_xlabel=True, show_ylabel=True):
    im = ax.imshow(matrix.values, vmin=0, vmax=1, cmap="RdYlGn", aspect="auto")
    ax.set_title(title, fontsize=10, fontweight="bold", loc="left", pad=8)
    ax.set_xticks(np.arange(matrix.shape[1]))
    ax.set_xticklabels([short_metric_labels[m] for m in matrix.columns], fontsize=7.2)
    if show_xlabel:
        ax.tick_params(axis="x", labelrotation=0)
    else:
        ax.set_xticklabels([])
    ax.set_yticks(np.arange(matrix.shape[0]))
    if show_ylabel:
        ax.set_yticklabels([short_model_labels.get(m, m) for m in matrix.index], fontsize=7.2)
    else:
        ax.set_yticklabels([])
    ax.set_xticks(np.arange(-0.5, matrix.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, matrix.shape[0], 1), minor=True)
    ax.grid(which="minor", color="white", linestyle="-", linewidth=0.75)
    ax.tick_params(which="minor", bottom=False, left=False)
    ax.tick_params(axis="both", length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    if annotate:
        for row_i in range(matrix.shape[0]):
            for col_i in range(matrix.shape[1]):
                value = matrix.iloc[row_i, col_i]
                if np.isfinite(value):
                    color = "white" if value < 0.22 or value > 0.78 else "#1f2937"
                    ax.text(col_i, row_i, f"{value:.2f}", ha="center", va="center", fontsize=6.2, color=color)
    return im


# Average-condition heatmap with values printed in each cell.
avg_matrix = standardized_heatmap_matrix(standardized_scores, "average", standardized_model_order, heatmap_plot_metrics)
fig, ax = plt.subplots(figsize=(10.8, 6.2))
im = add_standardized_heatmap(
    ax,
    avg_matrix,
    "Standardized Relative Performance: Average Across RT Timepoints",
    annotate=True,
)
cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Relative within-metric score (0 = weakest, 1 = strongest)", fontsize=8)
cbar.ax.tick_params(labelsize=7)
fig.text(
    0.02,
    0.015,
    "Scores are min-max standardized within each metric after aligning direction; raw values remain in the lollipop plots and CSV tables.",
    fontsize=7.5,
    color="#4C5663",
)
fig.tight_layout(rect=[0, 0.035, 1, 1])
std_avg_png = output_dir / "report_standardized_performance_heatmap_average.png"
std_avg_pdf = output_dir / "report_standardized_performance_heatmap_average.pdf"
std_avg_svg = output_dir / "report_standardized_performance_heatmap_average.svg"
fig.savefig(std_avg_png, dpi=300, bbox_inches="tight")
fig.savefig(std_avg_pdf, bbox_inches="tight")
fig.savefig(std_avg_svg, bbox_inches="tight")
plt.show()

# A4 portrait heatmap: IR-24h, IR-1week and average stacked.
fig, axes = plt.subplots(3, 1, figsize=(8.27, 11.69))
condition_titles = [("A", "IR-24h", "IR-24h"), ("B", "IR-1week", "IR-1week"), ("C", "average", "Average")]
last_im = None
for ax, (letter, condition_key, condition_label) in zip(axes, condition_titles):
    matrix = standardized_heatmap_matrix(standardized_scores, condition_key, standardized_model_order, heatmap_plot_metrics)
    last_im = add_standardized_heatmap(
        ax,
        matrix,
        f"{letter}. {condition_label}",
        annotate=False,
        show_xlabel=True,
        show_ylabel=True,
    )
fig.suptitle("Standardized Relative Performance Across Radiotherapy Timepoints", fontsize=13.2, fontweight="bold", y=0.982)
cbar = fig.colorbar(last_im, ax=axes, fraction=0.025, pad=0.018)
cbar.set_label("Relative within-metric score\n0 = weakest, 1 = strongest", fontsize=7.5)
cbar.ax.tick_params(labelsize=7)
fig.text(
    0.5,
    0.018,
    "Direction-aware min-max standardisation within each metric and timepoint. Use with raw lollipop plots to avoid over-reading small raw differences.",
    ha="center",
    fontsize=7.1,
    color="#4C5663",
)
fig.subplots_adjust(left=0.22, right=0.90, top=0.94, bottom=0.06, hspace=0.34)
std_a4_png = output_dir / "report_standardized_performance_heatmap_a4_portrait.png"
std_a4_pdf = output_dir / "report_standardized_performance_heatmap_a4_portrait.pdf"
std_a4_svg = output_dir / "report_standardized_performance_heatmap_a4_portrait.svg"
fig.savefig(std_a4_png, dpi=300)
fig.savefig(std_a4_pdf)
fig.savefig(std_a4_svg)
plt.show()

print(f"Saved standardized score table: {output_dir / 'report_standardized_performance_scores.csv'}")
print(f"Saved average standardized heatmap PNG: {std_avg_png}")
print(f"Saved A4 standardized heatmap PNG: {std_a4_png}")


# Z-score view: signed relative performance around the model mean for each
# metric and timepoint. Lower-is-better metrics are direction-aligned first.
from matplotlib.colors import TwoSlopeNorm


def compute_zscore_performance_scores(df, metrics):
    zscore_frames = []
    for condition in target_conditions + ["average"]:
        df_c = df[df["condition"] == condition].copy()
        z_df = df_c[["model"]].copy()
        for metric in metrics:
            values = df_c[metric].astype(float).replace([np.inf, -np.inf], np.nan)
            aligned = values if higher_is_better[metric] else -values
            mean_value = aligned.mean(skipna=True)
            std_value = aligned.std(skipna=True, ddof=0)
            if not np.isfinite(std_value) or std_value < 1e-12:
                z_df[metric] = 0.0
            else:
                z_df[metric] = (aligned - mean_value) / std_value
        z_df.insert(0, "condition", condition)
        zscore_frames.append(z_df)
    return pd.concat(zscore_frames, ignore_index=True)


zscore_scores = compute_zscore_performance_scores(summary_df, selected_plot_metrics)
zscore_scores.to_csv(output_dir / "report_zscore_performance_scores.csv", index=False)

z_average_scores = zscore_scores[zscore_scores["condition"] == "average"].copy()
z_average_scores["overall_zscore"] = z_average_scores[heatmap_plot_metrics].mean(axis=1)
zscore_model_order = [
    model
    for model in z_average_scores.sort_values("overall_zscore", ascending=False)["model"].tolist()
    if model not in excluded_heatmap_models
]


def zscore_heatmap_matrix(score_df, condition, model_order, metrics):
    matrix = score_df[score_df["condition"] == condition].set_index("model").reindex(model_order)[metrics]
    return matrix.astype(float)


global_z_abs = float(np.nanmax(np.abs(zscore_scores[selected_plot_metrics].to_numpy(dtype=float))))
global_z_abs = max(global_z_abs, 1.0)
z_norm = TwoSlopeNorm(vmin=-global_z_abs, vcenter=0.0, vmax=global_z_abs)


def add_zscore_heatmap(ax, matrix, title, annotate=False, show_xlabel=True, show_ylabel=True):
    im = ax.imshow(matrix.values, norm=z_norm, cmap="RdYlGn", aspect="auto")
    ax.set_title(title, fontsize=10, fontweight="bold", loc="left", pad=8)
    ax.set_xticks(np.arange(matrix.shape[1]))
    ax.set_xticklabels([short_metric_labels[m] for m in matrix.columns], fontsize=7.2)
    if show_xlabel:
        ax.tick_params(axis="x", labelrotation=0)
    else:
        ax.set_xticklabels([])
    ax.set_yticks(np.arange(matrix.shape[0]))
    if show_ylabel:
        ax.set_yticklabels([short_model_labels.get(m, m) for m in matrix.index], fontsize=7.2)
    else:
        ax.set_yticklabels([])
    ax.set_xticks(np.arange(-0.5, matrix.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, matrix.shape[0], 1), minor=True)
    ax.grid(which="minor", color="white", linestyle="-", linewidth=0.75)
    ax.tick_params(which="minor", bottom=False, left=False)
    ax.tick_params(axis="both", length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    if annotate:
        for row_i in range(matrix.shape[0]):
            for col_i in range(matrix.shape[1]):
                value = matrix.iloc[row_i, col_i]
                if np.isfinite(value):
                    color = "white" if abs(value) > global_z_abs * 0.55 else "#1f2937"
                    ax.text(col_i, row_i, f"{value:+.2f}", ha="center", va="center", fontsize=6.1, color=color)
    return im


# Average-condition z-score heatmap with signed values printed in each cell.
z_avg_matrix = zscore_heatmap_matrix(zscore_scores, "average", zscore_model_order, heatmap_plot_metrics)
fig, ax = plt.subplots(figsize=(10.8, 6.2))
im = add_zscore_heatmap(
    ax,
    z_avg_matrix,
    "Z-score Relative Performance: Average Across RT Timepoints",
    annotate=True,
)
cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Signed z-score after direction alignment\npositive = above model mean", fontsize=8)
cbar.ax.tick_params(labelsize=7)
fig.text(
    0.02,
    0.015,
    "Z-scores are computed within each metric after aligning direction; positive values indicate above-average relative performance for that metric.",
    fontsize=7.5,
    color="#4C5663",
)
fig.tight_layout(rect=[0, 0.04, 1, 1])
z_avg_png = output_dir / "report_zscore_performance_heatmap_average.png"
z_avg_pdf = output_dir / "report_zscore_performance_heatmap_average.pdf"
z_avg_svg = output_dir / "report_zscore_performance_heatmap_average.svg"
fig.savefig(z_avg_png, dpi=300, bbox_inches="tight")
fig.savefig(z_avg_pdf, bbox_inches="tight")
fig.savefig(z_avg_svg, bbox_inches="tight")
plt.show()

# A4 portrait z-score heatmap: IR-24h, IR-1week and average stacked.
fig, axes = plt.subplots(3, 1, figsize=(8.27, 11.69))
condition_titles = [("A", "IR-24h", "IR-24h"), ("B", "IR-1week", "IR-1week"), ("C", "average", "Average")]
last_im = None
for ax, (letter, condition_key, condition_label) in zip(axes, condition_titles):
    matrix = zscore_heatmap_matrix(zscore_scores, condition_key, zscore_model_order, heatmap_plot_metrics)
    last_im = add_zscore_heatmap(
        ax,
        matrix,
        f"{letter}. {condition_label}",
        annotate=False,
        show_xlabel=True,
        show_ylabel=True,
    )
fig.suptitle("Z-score Relative Performance Across Radiotherapy Timepoints", fontsize=13.2, fontweight="bold", y=0.982)
cbar = fig.colorbar(last_im, ax=axes, fraction=0.025, pad=0.018)
cbar.set_label("Signed z-score after direction alignment\npositive = above model mean", fontsize=7.5)
cbar.ax.tick_params(labelsize=7)
fig.text(
    0.5,
    0.018,
    "Signed z-scores are computed within each metric and timepoint after aligning metric direction. Negative values indicate below-average relative performance.",
    ha="center",
    fontsize=7.1,
    color="#4C5663",
)
fig.subplots_adjust(left=0.22, right=0.90, top=0.94, bottom=0.06, hspace=0.34)
z_a4_png = output_dir / "report_zscore_performance_heatmap_a4_portrait.png"
z_a4_pdf = output_dir / "report_zscore_performance_heatmap_a4_portrait.pdf"
z_a4_svg = output_dir / "report_zscore_performance_heatmap_a4_portrait.svg"
fig.savefig(z_a4_png, dpi=300)
fig.savefig(z_a4_pdf)
fig.savefig(z_a4_svg)
plt.show()

print(f"Saved z-score performance table: {output_dir / 'report_zscore_performance_scores.csv'}")
print(f"Saved average z-score heatmap PNG: {z_avg_png}")
print(f"Saved A4 z-score heatmap PNG: {z_a4_png}")

print(f"\nSaved refined analysis outputs to: {output_dir.resolve()}")
print("Interpretation frame:")
print("- Condition mean and condition-only classical models are sanity/comparator baselines.")
print("- scGPT is evaluated from the Myriad final test_results.npz output.")
print("- Local E-distance is reported for all genes only in the report-facing plots.")
print("- Spearman and Common-DEGs are omitted from report-facing outputs to avoid redundant interpretation.")
print("- DE-specific conclusions should rely on Pearson DE Delta, AUPRC, and DEG visualisations.")


## Published Benchmark Context

This section does not claim a direct numerical leaderboard comparison, because the cited studies use different open-source perturbation datasets, usually genetic or drug perturbation screens. The purpose is to compare the *pattern* of our benchmarking result against the broader literature: whether simple baselines are competitive, whether metric choice changes model ranking, and whether foundation models show clear added value beyond classical approaches.


In [ ]:
# ==========================================================
# 9. Contextual comparison with published perturbation benchmarks
# ==========================================================
published_benchmarks = pd.DataFrame([
    {
        "study": "Ahlmann-Eltze et al. 2025, Nature Methods",
        "source": "https://www.nature.com/articles/s41592-025-02772-6",
        "datasets": "Norman CRISPRa; Replogle CRISPRi; Adamson Perturb-seq",
        "models_or_baselines": "scGPT, scFoundation, GEARS, CPA, Geneformer, UCE, scBERT; no-change, mean, additive and linear baselines",
        "metric_emphasis": "L2/error metrics and Pearson Delta over selected/readout genes",
        "main_message": "Deep perturbation models did not consistently outperform deliberately simple baselines in their benchmark setup.",
        "comparison_to_this_project": "Directly supports our decision to benchmark scGPT against condition-mean and classical baselines before claiming added value.",
    },
    {
        "study": "Csendes et al. 2025, BMC Genomics",
        "source": "https://pubmed.ncbi.nlm.nih.gov/40269681/",
        "datasets": "Adamson; Norman; Replogle K562; Replogle RPE1",
        "models_or_baselines": "scGPT, scFoundation, train-mean baseline and Random Forest models with biological features",
        "metric_emphasis": "Pearson Delta for post-perturbation RNA-seq prediction",
        "main_message": "The train-mean baseline outperformed scGPT/scFoundation, and biologically informed basic ML models outperformed scGPT by a large margin.",
        "comparison_to_this_project": "Closest conceptual match to our notebook: compare scGPT with train/condition-mean and Random Forest-type baselines, while noting our perturbation is radiotherapy rather than CRISPR.",
    },
    {
        "study": "Bendidi et al. 2024, arXiv",
        "source": "https://arxiv.org/abs/2410.13956",
        "datasets": "Diverse public transcriptomic perturbation datasets across sequencing technologies and cell lines",
        "models_or_baselines": "Transcriptomics foundation models compared with PCA/scVI and classical representation approaches",
        "metric_emphasis": "Biologically motivated perturbation-analysis tasks rather than one global score",
        "main_message": "PCA and scVI were better suited than several foundation models for perturbation-analysis tasks in their framework.",
        "comparison_to_this_project": "Suggests adding a PCA-ridge or PCA-regression baseline as a strong, interpretable classical comparator.",
    },
    {
        "study": "Wu et al. 2024/2025, PerturBench preprint",
        "source": "https://arxiv.org/abs/2408.10609",
        "datasets": "Norman19; Srivatsan20; Frangieh21; Jiang24; McFaline-Figueroa23",
        "models_or_baselines": "Published perturbation models plus linear, latent-additive and decoder-only baselines",
        "metric_emphasis": "RMSE/cosine plus rank metrics and distributional checks for mode collapse",
        "main_message": "Model rankings depend on split design and metric choice; simpler architectures are often competitive and scale well.",
        "comparison_to_this_project": "Supports keeping several complementary metric families and adding a stronger control-matched linear/ridge baseline if time allows.",
    },
])

our_dataset_context = pd.DataFrame([
    {
        "comparison_axis": "Biological system",
        "this_project": "Murine GBM single-cell RNA-seq radiotherapy dataset",
        "published_benchmarks": "Mostly K562/RPE1/cell-line or pooled perturbation-screen datasets",
        "interpretation": "Our setting is clinically closer to GBM radiotherapy biology but less suited to direct leaderboard comparison.",
    },
    {
        "comparison_axis": "Perturbation type",
        "this_project": "Ionising radiation at IR-24h and IR-1week",
        "published_benchmarks": "Mostly CRISPR/CRISPRi/CRISPRa or drug perturbations",
        "interpretation": "Radiotherapy is a broad stressor, so model performance should be interpreted as response-state prediction rather than single-target perturbation prediction.",
    },
    {
        "comparison_axis": "Comparable metrics",
        "this_project": "Pearson, Pearson Delta, Pearson DE Delta, AUPRC and full-gene distributional distances",
        "published_benchmarks": "Ahlmann/Csendes emphasise Pearson Delta and error-style metrics; PerturBench adds rank/distributional checks",
        "interpretation": "Pearson Delta is the most direct shared language; AUPRC and distributional metrics add biology-aware and population-level interpretation.",
    },
    {
        "comparison_axis": "Best comparison logic",
        "this_project": "Ask whether scGPT adds value beyond condition-mean/classical baselines on GBM radiotherapy data",
        "published_benchmarks": "Ask whether foundation/deep models beat simple baselines across public perturbation screens",
        "interpretation": "The strongest claim is dataset-specific: whether scGPT is useful in this GBM radiotherapy setting, not whether it universally beats classical models.",
    },
])

suggested_extensions = pd.DataFrame([
    {
        "category": "Metric",
        "suggestion": "RMSE or L2 error on all genes and on highly variable / highly expressed genes",
        "why_it_helps": "Ahlmann-style benchmarks use error/L2-style metrics; adding RMSE gives a simple lower-is-better comparator alongside Pearson.",
        "priority": "High",
    },
    {
        "category": "Metric",
        "suggestion": "Variance-compression ratio for predicted vs observed deltas",
        "why_it_helps": "Checks whether a model predicts the correct direction but compresses the magnitude of radiation-induced response.",
        "priority": "Medium",
    },
    {
        "category": "Metric",
        "suggestion": "Pathway-level delta agreement for DNA damage, oxidative stress, hypoxia, cell-cycle and immune pathways",
        "why_it_helps": "Adds GBM/radiotherapy biological relevance beyond gene-wise statistics.",
        "priority": "Medium",
    },
    {
        "category": "Model",
        "suggestion": "PCA-ridge / PCA-linear baseline",
        "why_it_helps": "Bendidi et al. found PCA-style representations competitive with transcriptomic foundation models; this is a strong, explainable comparator.",
        "priority": "High",
    },
    {
        "category": "Model",
        "suggestion": "Control-matched ridge baseline using sampled control cells plus condition/timepoint encoding",
        "why_it_helps": "Closer to PerturBench-style counterfactual prediction than condition-only lookup models, because it predicts perturbed states from control-cell states.",
        "priority": "High",
    },
    {
        "category": "Model",
        "suggestion": "Random Forest or gradient boosting with biological/QC/cell-state features",
        "why_it_helps": "Csendes et al. found biologically informed basic ML competitive; useful if metadata contains subtype, sample, QC, PCA or cell-state covariates.",
        "priority": "Medium",
    },
])

published_benchmarks.to_csv(output_dir / "published_benchmark_context.csv", index=False)
our_dataset_context.to_csv(output_dir / "our_dataset_vs_published_benchmarks.csv", index=False)
suggested_extensions.to_csv(output_dir / "suggested_next_metrics_models.csv", index=False)

print("Published benchmark papers for contextual comparison:")
display(published_benchmarks)

print("How this GBM radiotherapy project differs from open-source benchmark datasets:")
display(our_dataset_context)

print("Suggested next metrics / benchmark models:")
display(suggested_extensions)

# Compact visual summary for the report/appendix.
fig, ax = plt.subplots(figsize=(13, 5.2))
ax.axis("off")
wrapped = published_benchmarks[["study", "models_or_baselines", "metric_emphasis", "main_message", "comparison_to_this_project"]].copy()
table = ax.table(
    cellText=wrapped.values,
    colLabels=["Study", "Models / baselines", "Metric emphasis", "Main benchmark message", "How to compare with ours"],
    loc="center",
    cellLoc="left",
)
table.auto_set_font_size(False)
table.set_fontsize(7)
table.scale(1, 2.0)
for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor("#d0d4d9")
    if row == 0:
        cell.set_facecolor("#e8eef6")
        cell.set_text_props(weight="bold")
    else:
        cell.set_facecolor("#ffffff" if row % 2 else "#f7f9fb")
plt.tight_layout()
plt.savefig(output_dir / "published_benchmark_context_table.png", dpi=300, bbox_inches="tight")
plt.show()


## Manuscript Literature Comparison Figures

This report-facing comparison places this glioblastoma radiotherapy benchmark beside selected published perturbation-prediction benchmarks and exports two separate figures: a benchmark-value table and a Pearson Delta illustration. The wording is deliberately cautious because the datasets, perturbation types, split definitions, and task settings are not identical across studies.

The **This Glioblastoma Radiotherapy Project** rows in the table below are generated from this notebook's computed GB RT metrics, rather than from manually supplied placeholder values. Published-study rows remain manually extracted from Ahlmann-Eltze et al. 2025 and Csendes et al. 2025.


In [ ]:
# ==========================================================
# Manuscript-ready literature comparison table + separate report figures
# ==========================================================
from pathlib import Path
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

output_dir = Path('refined_baseline_outputs')
output_dir.mkdir(exist_ok=True)

def format_metric_value(value):
    if pd.isna(value):
        return "Not computed"
    return f"{float(value):.3f}"


def load_project_average_metrics():
    if "avg_report_df" in globals():
        return avg_report_df.copy()
    return pd.read_csv(output_dir / "report_focused_average_metrics.csv")


project_average_metrics = load_project_average_metrics()

# Use the current post-hoc scGPT final metrics where available. Pearson DE
# remains sourced from the benchmark table because it is not in this summary.
scgpt_report_path = Path('analysis_outputs/scgpt_model_evaluation/test_report_metrics.csv')
if scgpt_report_path.exists():
    scgpt_report = pd.read_csv(scgpt_report_path)
    scgpt_average = scgpt_report.loc[scgpt_report['condition'] == 'average']
    scgpt_mask = project_average_metrics['model'].eq('scGPT')
    if not scgpt_average.empty and scgpt_mask.any():
        scgpt_average = scgpt_average.iloc[0]
        project_average_metrics.loc[scgpt_mask, 'pearson'] = scgpt_average['Pearson']
        project_average_metrics.loc[scgpt_mask, 'pearson_delta'] = scgpt_average['Delta']
        project_average_metrics.loc[scgpt_mask, 'pearson_delta_de'] = scgpt_average['DE_Delta']
project_model_rows = [
    {
        "source_model": "scGPT",
        "display_model": "scGPT",
        "model_class": "scGPT",
        "interpretation": "Computed average values are high across Pearson-family metrics; condition-only baselines remain competitive on mean-correlation metrics.",
    },
    {
        "source_model": "Seen-condition mean",
        "display_model": "Seen-condition mean",
        "model_class": "Mean baseline",
        "interpretation": "Computed values suggest this condition-average baseline is highly competitive on mean-profile metrics.",
    },
    {
        "source_model": "Elastic Net PCA delta",
        "display_model": "Elastic Net PCA delta",
        "model_class": "Classical / linear",
        "interpretation": "PCA-constrained linear modelling appears competitive, though below the strongest condition/RF baselines on Pearson Delta.",
    },
    {
        "source_model": "Bilinear Ridge delta",
        "display_model": "Bilinear Ridge delta",
        "model_class": "Classical / linear",
        "interpretation": "Low-rank ridge structure appears competitive on Pearson-family metrics in this computed summary.",
    },
    {
        "source_model": "Pseudobulk RF-PCA",
        "display_model": "Pseudobulk RF-PCA",
        "model_class": "Classical / nonlinear",
        "interpretation": "The saved-split values suggest competitive delta reconstruction, while the pseudobulk design provides a more informative RF comparator than condition-only encoding.",
    },
    {
        "source_model": "KNN delta",
        "display_model": "KNN delta",
        "model_class": "Classical / neighbor",
        "interpretation": "KNN shows lower Pearson-family metrics than the strongest selected project baselines in this average summary.",
    },
]

gb_project_rows = []
for spec in project_model_rows:
    match = project_average_metrics[project_average_metrics["model"] == spec["source_model"]]
    if match.empty:
        continue
    row = match.iloc[0]
    gb_project_rows.append({
        "Study": "This Glioblastoma Radiotherapy Project",
        "Dataset / setting": "Glioblastoma, ctrl vs IR-24h / IR-1week" if not gb_project_rows else "Same",
        "Model": spec["display_model"],
        "Model class": spec["model_class"],
        "Pearson": format_metric_value(row.get("pearson", np.nan)),
        "Pearson DE": format_metric_value(row.get("pearson_de", np.nan)),
        "Pearson Delta": format_metric_value(row.get("pearson_delta", np.nan)),
        "Pearson Delta DE": format_metric_value(row.get("pearson_delta_de", np.nan)),
        "Interpretation": spec["interpretation"],
    })

literature_rows = [
    {'Study': 'Ahlmann-Eltze et al., 2025', 'Dataset / setting': 'Adamson, Replogle K562/RPE1; unseen single-gene perturbations', 'Model': 'scGPT', 'Model class': 'scGPT', 'Pearson': '0.975', 'Pearson DE': 'Not reported', 'Pearson Delta': '0.486', 'Pearson Delta DE': 'Not reported', 'Interpretation': 'scGPT appears lower than simple baselines on the reported single-gene task.'},
    {'Study': 'Ahlmann-Eltze et al., 2025', 'Dataset / setting': 'Same', 'Model': 'Mean baseline', 'Model class': 'Mean baseline', 'Pearson': '0.978', 'Pearson DE': 'Not reported', 'Pearson Delta': '0.575', 'Pearson Delta DE': 'Not reported', 'Interpretation': 'Mean baseline shows higher Pearson Delta than scGPT in this summary.'},
    {'Study': 'Ahlmann-Eltze et al., 2025', 'Dataset / setting': 'Same', 'Model': 'Self-trained linear/PCA model', 'Model class': 'Classical / linear', 'Pearson': '0.979', 'Pearson DE': 'Not reported', 'Pearson Delta': '0.586', 'Pearson Delta DE': 'Not reported', 'Interpretation': 'Values point toward the self-trained linear/PCA model as most favorable among these rows.'},
    {'Study': 'Csendes et al., 2025', 'Dataset / setting': 'Adamson, Norman, Replogle K562/RPE1; CRISPR Perturb-seq', 'Model': 'scGPT', 'Model class': 'scGPT', 'Pearson': '0.985', 'Pearson DE': '0.950', 'Pearson Delta': '0.529', 'Pearson Delta DE': '0.666', 'Interpretation': 'The reported values do not suggest consistent scGPT advantage over baselines on delta metrics.'},
    {'Study': 'Csendes et al., 2025', 'Dataset / setting': 'Same', 'Model': 'Train mean', 'Model class': 'Mean baseline', 'Pearson': '0.986', 'Pearson DE': '0.933', 'Pearson Delta': '0.567', 'Pearson Delta DE': '0.618', 'Interpretation': 'Mean baseline shows higher Pearson Delta than scGPT in this summary.'},
    {'Study': 'Csendes et al., 2025', 'Dataset / setting': 'Same', 'Model': 'RF-GO', 'Model class': 'Classical / linear', 'Pearson': '0.987', 'Pearson DE': '0.942', 'Pearson Delta': '0.613', 'Pearson Delta DE': '0.669', 'Interpretation': 'Values point toward GO-informed RF as the most favorable listed Csendes baseline.'},
    {'Study': 'Csendes et al., 2025', 'Dataset / setting': 'Same', 'Model': 'RF-scGPT embeddings', 'Model class': 'Embedding baseline', 'Pearson': '0.987', 'Pearson DE': '0.940', 'Pearson Delta': '0.591', 'Pearson Delta DE': '0.637', 'Interpretation': 'scGPT embeddings may be useful for RF, though not clearly best overall.'},
    {'Study': 'Csendes et al., 2025', 'Dataset / setting': 'Same', 'Model': 'kNN-GO', 'Model class': 'Biology-informed neighbor', 'Pearson': '0.987', 'Pearson DE': '0.944', 'Pearson Delta': '0.597', 'Pearson Delta DE': '0.659', 'Interpretation': 'kNN with biological features appears competitive.'},
]

manuscript_table = pd.DataFrame(gb_project_rows + literature_rows)

excluded_figure_models = {'Elastic Net PCA delta', 'KNN delta', 'kNN-GO'}
manuscript_table = (
    manuscript_table
    .loc[~manuscript_table['Model'].isin(excluded_figure_models)]
    .reset_index(drop=True)
)

display_cols = ['Study', 'Dataset / setting', 'Model', 'Pearson', 'Pearson Delta', 'Pearson Delta DE', 'Interpretation']
plot_table = manuscript_table[display_cols].copy()

csv_path = output_dir / 'manuscript_literature_comparison_table.csv'
figure1_table_png_path = output_dir / 'figure_1_literature_comparison_table.png'
figure1_table_pdf_path = output_dir / 'figure_1_literature_comparison_table.pdf'
figure1_table_svg_path = output_dir / 'figure_1_literature_comparison_table.svg'
figure2_delta_png_path = output_dir / 'figure_2_pearson_delta_illustration.png'
figure2_delta_pdf_path = output_dir / 'figure_2_pearson_delta_illustration.pdf'
figure2_delta_svg_path = output_dir / 'figure_2_pearson_delta_illustration.svg'
plot_table.to_csv(csv_path, index=False)

display(plot_table)

palette = {
    'scGPT': '#2D5A87',
    'Mean baseline': '#7B858C',
    'Classical / linear': '#3F8C5A',
    'Classical / nonlinear': '#2A9D8F',
    'Classical / neighbor': '#6C5BA7',
    'Embedding baseline': '#C47A36',
    'Biology-informed neighbor': '#8A6BBE',
}
group_shades = {
    'This Glioblastoma Radiotherapy Project': '#E9F4F3',
    'Ahlmann-Eltze et al., 2025': '#F3F0E8',
    'Csendes et al., 2025': '#EEF1F6',
}
header_color = '#22384F'
grid_color = '#D8DEE8'

short_study = {
    'This Glioblastoma Radiotherapy Project': 'This Glioblastoma Radiotherapy Project',
    'Ahlmann-Eltze et al., 2025': 'Ahlmann-Eltze 2025',
    'Csendes et al., 2025': 'Csendes 2025',
}

def wrap_frame(df, widths):
    wrapped = df.copy()
    for col, width in widths.items():
        wrapped[col] = wrapped[col].map(lambda value: textwrap.fill(str(value), width=width))
    return wrapped

def add_table(ax, df, title=None, bbox=(0, 0, 1, 1), fontsize=7.4):
    wrapped = wrap_frame(df, {
        'Study': 24,
        'Dataset / setting': 34,
        'Model': 24,
        'Pearson': 12,
        'Pearson Delta': 13,
        'Pearson Delta DE': 15,
        'Interpretation': 44,
    })
    ax.axis('off')
    if title:
        ax.set_title(title, fontsize=17, fontweight='bold', loc='left', pad=10)
    table = ax.table(
        cellText=wrapped.values,
        colLabels=wrapped.columns,
        loc='center',
        cellLoc='left',
        colLoc='left',
        colWidths=[0.15, 0.22, 0.14, 0.07, 0.09, 0.10, 0.24],
        bbox=bbox,
    )
    table.auto_set_font_size(False)
    table.set_fontsize(fontsize)
    for (row, col), cell in table.get_celld().items():
        cell.set_edgecolor(grid_color)
        if row == 0:
            cell.set_facecolor(header_color)
            cell.set_text_props(color='white', weight='bold')
        else:
            study = df.iloc[row - 1]['Study']
            prev_study = df.iloc[row - 2]['Study'] if row > 1 else None
            if study != prev_study:
                cell.set_facecolor(group_shades.get(study, '#F6F7F9'))
                if col == 0:
                    cell.set_text_props(weight='bold')
            elif row % 2 == 0:
                cell.set_facecolor('#FAFBFC')
            else:
                cell.set_facecolor('white')
    return table

# Figure 1: standalone report table.
fig_table, ax_table = plt.subplots(figsize=(22, 8.4))
add_table(
    ax_table,
    plot_table,
    title='Literature Benchmark Values for This Glioblastoma Radiotherapy Project',
    bbox=[0, 0, 1, 0.94],
    fontsize=7.0,
)
fig_table.text(
    0.5, 0.015,
    textwrap.fill(
        'Note: Values are compared across non-identical datasets and settings; interpretation is therefore directional rather than inferential.',
        width=150,
    ),
    ha='center',
    multialignment='center',
    fontsize=8.5,
    color='#4C5663',
)
fig_table.savefig(figure1_table_png_path, dpi=240, bbox_inches='tight')
fig_table.savefig(figure1_table_pdf_path, bbox_inches='tight')
fig_table.savefig(figure1_table_svg_path, bbox_inches='tight')
plt.show()

# Shared data preparation for Figure 2.
plot_df = manuscript_table.copy()
plot_df['Pearson Delta numeric'] = pd.to_numeric(plot_df['Pearson Delta'], errors='coerce')
plot_df['Short study'] = plot_df['Study'].map(short_study)
plot_df['Plot label'] = plot_df['Short study'] + ' | ' + plot_df['Model']
plot_df = plot_df.dropna(subset=['Pearson Delta numeric']).reset_index(drop=True)
plot_df[['Study', 'Dataset / setting', 'Model', 'Model class', 'Pearson Delta numeric']].to_csv(
    output_dir / 'figure_2_pearson_delta_illustration_source_data.csv', index=False
)
plot_df = plot_df.iloc[::-1].reset_index(drop=True)
y = np.arange(len(plot_df))

# Figure 2: standalone Pearson Delta illustration.
fig_delta, ax_delta = plt.subplots(figsize=(16.5, max(7.0, 0.42 * len(plot_df))))

for study, group in plot_df.groupby('Study', sort=False):
    ymin = y[group.index.min()] - 0.48
    ymax = y[group.index.max()] + 0.48
    ax_delta.axhspan(ymin, ymax, color=group_shades.get(study, '#F6F7F9'), alpha=0.62, zorder=0)

for idx, row in plot_df.iterrows():
    color = palette.get(row['Model class'], '#555555')
    value = row['Pearson Delta numeric']
    ax_delta.plot([0, value], [idx, idx], color=color, lw=2.35, alpha=0.55, solid_capstyle='round', zorder=2)
    ax_delta.scatter(value, idx, s=104 if row['Model class'] == 'scGPT' else 78, color=color, edgecolor='white', linewidth=1.15, zorder=3)
    ax_delta.text(value + 0.012, idx, f'{value:.3f}', va='center', ha='left', fontsize=9, color='#26313D')

ax_delta.set_yticks(y)
ax_delta.set_yticklabels(plot_df['Plot label'], fontsize=8.7)
max_delta_value = float(plot_df['Pearson Delta numeric'].max())
ax_delta.set_xlim(0, min(1.05, max_delta_value + 0.055))
ax_delta.set_xlabel('Pearson Delta (higher is better)', fontsize=10.8)
ax_delta.set_title('Pearson Delta Context Across Studies', fontsize=16, fontweight='bold', loc='left', pad=12)
ax_delta.grid(axis='x', color=grid_color, lw=0.85, alpha=0.82)
ax_delta.spines[['top', 'right', 'left']].set_visible(False)
ax_delta.tick_params(axis='y', length=0)

used_model_classes = [label for label in palette if label in set(plot_df['Model class'])]
handles = [
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette[label], markeredgecolor='white', markersize=8, label=label)
    for label in used_model_classes
]
ax_delta.legend(handles=handles, ncol=3, frameon=False, bbox_to_anchor=(0, -0.18, 1, 0.1), loc='upper left', fontsize=8.8)
fig_delta.text(
    0.5,
    0.015,
    (
        'Sources: Ahlmann-Eltze et al. 2025 Source Data Fig. 2; Csendes et al. 2025 Supplementary Table 2; '
        'computed notebook values for this GB radiotherapy benchmark.\n'
        'Cross-study comparisons are descriptive because datasets and perturbation settings differ.'
    ),
    ha='center',
    multialignment='center',
    fontsize=8.3,
    color='#4C5663',
)
fig_delta.subplots_adjust(left=0.36, right=0.985, top=0.88, bottom=0.23)
fig_delta.savefig(figure2_delta_png_path, dpi=240, bbox_inches='tight')
fig_delta.savefig(figure2_delta_pdf_path, bbox_inches='tight')
fig_delta.savefig(figure2_delta_svg_path, bbox_inches='tight')
plt.show()

print(f'Saved manuscript table CSV: {csv_path}')
print(f'Saved Figure 1 table PNG: {figure1_table_png_path}')
print(f'Saved Figure 1 table PDF: {figure1_table_pdf_path}')
print(f'Saved Figure 1 table SVG: {figure1_table_svg_path}')
print(f'Saved Figure 2 Pearson Delta PNG: {figure2_delta_png_path}')
print(f'Saved Figure 2 Pearson Delta PDF: {figure2_delta_pdf_path}')
print(f'Saved Figure 2 Pearson Delta SVG: {figure2_delta_svg_path}')


## Extracted Benchmark Values Compared with This Project

This section extracts numerical benchmark values from Ahlmann-Eltze et al. and Csendes et al. and compares them with the average GBM radiotherapy benchmark metrics from this notebook. The comparison is not a direct leaderboard because the published papers use public CRISPR perturbation datasets, whereas this project uses GBM radiotherapy timepoints. The most comparable shared metric is Pearson Delta.


In [ ]:
# ==========================================================
# 10. Extracted published values compared with this project
# ==========================================================
# Values below were extracted from:
# - Ahlmann-Eltze et al. 2025 Nature Methods, Source Data Fig. 2, Panel A.
# - Csendes et al. 2025 BMC Genomics, Supplementary Table 2.
# - This notebook's report_focused_average_metrics.csv.

# Ahlmann source-data columns are named r2 and r2_delta. They correspond to
# raw expression-profile correlation and Pearson Delta-style correlation.
ahlmann_rows = [
    # Dataset-level values aggregated from Source Data Fig. 2 Panel A, test rows.
    {"study": "Ahlmann-Eltze 2025", "dataset": "Adamson", "model_group": "scGPT", "model": "scGPT", "pearson": 0.980650, "pearson_delta": 0.568668, "pearson_delta_de": np.nan, "l2": 5.053416, "source_detail": "Source Data Fig. 2 Panel A; mean over test perturbations"},
    {"study": "Ahlmann-Eltze 2025", "dataset": "Adamson", "model_group": "Simple mean", "model": "mean", "pearson": 0.986262, "pearson_delta": 0.687619, "pearson_delta_de": np.nan, "l2": 4.081029, "source_detail": "Source Data Fig. 2 Panel A; mean over test perturbations"},
    {"study": "Ahlmann-Eltze 2025", "dataset": "Adamson", "model_group": "Linear/classical", "model": "linear perturbation model", "pearson": 0.984903, "pearson_delta": 0.675790, "pearson_delta_de": np.nan, "l2": 4.285925, "source_detail": "Source Data Fig. 2 Panel A; mean over test perturbations"},
    {"study": "Ahlmann-Eltze 2025", "dataset": "Replogle K562", "model_group": "scGPT", "model": "scGPT", "pearson": 0.983802, "pearson_delta": 0.316710, "pearson_delta_de": np.nan, "l2": 4.335095, "source_detail": "Source Data Fig. 2 Panel A; mean over test perturbations"},
    {"study": "Ahlmann-Eltze 2025", "dataset": "Replogle K562", "model_group": "Simple mean", "model": "mean", "pearson": 0.984648, "pearson_delta": 0.403887, "pearson_delta_de": np.nan, "l2": 4.118693, "source_detail": "Source Data Fig. 2 Panel A; mean over test perturbations"},
    {"study": "Ahlmann-Eltze 2025", "dataset": "Replogle K562", "model_group": "Linear/classical", "model": "linear perturbation model", "pearson": 0.985862, "pearson_delta": 0.432395, "pearson_delta_de": np.nan, "l2": 3.996537, "source_detail": "Source Data Fig. 2 Panel A; mean over test perturbations"},
    {"study": "Ahlmann-Eltze 2025", "dataset": "Replogle RPE1", "model_group": "scGPT", "model": "scGPT", "pearson": 0.969416, "pearson_delta": 0.585005, "pearson_delta_de": np.nan, "l2": 6.052112, "source_detail": "Source Data Fig. 2 Panel A; mean over test perturbations"},
    {"study": "Ahlmann-Eltze 2025", "dataset": "Replogle RPE1", "model_group": "Simple mean", "model": "mean", "pearson": 0.973236, "pearson_delta": 0.671878, "pearson_delta_de": np.nan, "l2": 5.609620, "source_detail": "Source Data Fig. 2 Panel A; mean over test perturbations"},
    {"study": "Ahlmann-Eltze 2025", "dataset": "Replogle RPE1", "model_group": "Linear/classical", "model": "linear perturbation model", "pearson": 0.974359, "pearson_delta": 0.674900, "pearson_delta_de": np.nan, "l2": 5.469767, "source_detail": "Source Data Fig. 2 Panel A; mean over test perturbations"},
    # Overall test-perturbation weighted means across the three Ahlmann single-perturbation datasets.
    {"study": "Ahlmann-Eltze 2025", "dataset": "Overall", "model_group": "scGPT", "model": "scGPT", "pearson": 0.975375, "pearson_delta": 0.486069, "pearson_delta_de": np.nan, "l2": 5.362476, "source_detail": "Source Data Fig. 2 Panel A; weighted by available test perturbation rows"},
    {"study": "Ahlmann-Eltze 2025", "dataset": "Overall", "model_group": "Simple mean", "model": "mean", "pearson": 0.978223, "pearson_delta": 0.575079, "pearson_delta_de": np.nan, "l2": 4.969020, "source_detail": "Source Data Fig. 2 Panel A; weighted by available test perturbation rows"},
    {"study": "Ahlmann-Eltze 2025", "dataset": "Overall", "model_group": "Linear/classical", "model": "linear perturbation model", "pearson": 0.979223, "pearson_delta": 0.586463, "pearson_delta_de": np.nan, "l2": 4.857398, "source_detail": "Source Data Fig. 2 Panel A; weighted by available test perturbation rows"},
]

csendes_rows = [
    # Values extracted from Csendes et al. Supplementary Table 2.
    {"study": "Csendes 2025", "dataset": "Adamson", "model_group": "scGPT", "model": "scGPT", "pearson": 0.990404, "pearson_delta": 0.641038, "pearson_delta_de": 0.797411, "l2": np.nan, "source_detail": "Supplementary Table 2"},
    {"study": "Csendes 2025", "dataset": "Adamson", "model_group": "Simple mean", "model": "mean", "pearson": 0.992586, "pearson_delta": 0.711148, "pearson_delta_de": 0.729265, "l2": np.nan, "source_detail": "Supplementary Table 2"},
    {"study": "Csendes 2025", "dataset": "Adamson", "model_group": "Linear/classical", "model": "RF_go", "pearson": 0.992508, "pearson_delta": 0.738712, "pearson_delta_de": 0.741601, "l2": np.nan, "source_detail": "Supplementary Table 2"},
    {"study": "Csendes 2025", "dataset": "Norman", "model_group": "scGPT", "model": "scGPT", "pearson": 0.987689, "pearson_delta": 0.553838, "pearson_delta_de": 0.731822, "l2": np.nan, "source_detail": "Supplementary Table 2"},
    {"study": "Csendes 2025", "dataset": "Norman", "model_group": "Simple mean", "model": "mean", "pearson": 0.986273, "pearson_delta": 0.557266, "pearson_delta_de": 0.642254, "l2": np.nan, "source_detail": "Supplementary Table 2"},
    {"study": "Csendes 2025", "dataset": "Norman", "model_group": "Linear/classical", "model": "RF_go", "pearson": 0.987946, "pearson_delta": 0.586124, "pearson_delta_de": 0.668596, "l2": np.nan, "source_detail": "Supplementary Table 2"},
    {"study": "Csendes 2025", "dataset": "Replogle K562", "model_group": "scGPT", "model": "scGPT", "pearson": 0.986341, "pearson_delta": 0.326502, "pearson_delta_de": 0.480966, "l2": np.nan, "source_detail": "Supplementary Table 2"},
    {"study": "Csendes 2025", "dataset": "Replogle K562", "model_group": "Simple mean", "model": "mean", "pearson": 0.987384, "pearson_delta": 0.372996, "pearson_delta_de": 0.460378, "l2": np.nan, "source_detail": "Supplementary Table 2"},
    {"study": "Csendes 2025", "dataset": "Replogle K562", "model_group": "Linear/classical", "model": "RF_go", "pearson": 0.989273, "pearson_delta": 0.480000, "pearson_delta_de": 0.597284, "l2": np.nan, "source_detail": "Supplementary Table 2"},
    {"study": "Csendes 2025", "dataset": "Replogle RPE1", "model_group": "scGPT", "model": "scGPT", "pearson": 0.974120, "pearson_delta": 0.596358, "pearson_delta_de": 0.652319, "l2": np.nan, "source_detail": "Supplementary Table 2"},
    {"study": "Csendes 2025", "dataset": "Replogle RPE1", "model_group": "Simple mean", "model": "mean", "pearson": 0.976092, "pearson_delta": 0.628385, "pearson_delta_de": 0.639987, "l2": np.nan, "source_detail": "Supplementary Table 2"},
    {"study": "Csendes 2025", "dataset": "Replogle RPE1", "model_group": "Linear/classical", "model": "RF_go", "pearson": 0.979307, "pearson_delta": 0.647720, "pearson_delta_de": 0.670487, "l2": np.nan, "source_detail": "Supplementary Table 2"},
    {"study": "Csendes 2025", "dataset": "Overall", "model_group": "scGPT", "model": "scGPT", "pearson": 0.984638, "pearson_delta": 0.529434, "pearson_delta_de": 0.665629, "l2": np.nan, "source_detail": "Mean across Supplementary Table 2 datasets"},
    {"study": "Csendes 2025", "dataset": "Overall", "model_group": "Simple mean", "model": "mean", "pearson": 0.985584, "pearson_delta": 0.567449, "pearson_delta_de": 0.617971, "l2": np.nan, "source_detail": "Mean across Supplementary Table 2 datasets"},
    {"study": "Csendes 2025", "dataset": "Overall", "model_group": "Linear/classical", "model": "RF_go", "pearson": 0.987259, "pearson_delta": 0.613139, "pearson_delta_de": 0.669492, "l2": np.nan, "source_detail": "Mean across Supplementary Table 2 datasets"},
]

published_values_df = pd.DataFrame(ahlmann_rows + csendes_rows)

our_avg = pd.read_csv(output_dir / "report_focused_average_metrics.csv")
our_model_map = {
    "scGPT": "scGPT",
    "Seen-condition mean": "Simple mean",
    "Pseudobulk RF-PCA": "Linear/classical",
}
our_rows = []
for model, group in our_model_map.items():
    row = our_avg[our_avg["model"] == model].iloc[0]
    our_rows.append({
        "study": "This project",
        "dataset": "GBM radiotherapy average",
        "model_group": group,
        "model": model,
        "pearson": float(row["pearson"]),
        "pearson_delta": float(row["pearson_delta"]),
        "pearson_delta_de": float(row["pearson_delta_de"]),
        "l2": np.nan,
        "source_detail": "Classical_Models_v5 report_focused_average_metrics.csv",
    })
our_values_df = pd.DataFrame(our_rows)

published_vs_ours_df = pd.concat([published_values_df, our_values_df], ignore_index=True)
published_vs_ours_df.to_csv(output_dir / "published_vs_ours_extracted_metric_values.csv", index=False)

print("Extracted published values plus our GBM radiotherapy results:")
display(published_vs_ours_df)

# Overall-comparison table: one row per study and model group.
overall_compare_df = published_vs_ours_df[
    published_vs_ours_df["dataset"].isin(["Overall", "GBM radiotherapy average"])
].copy()
overall_compare_df.to_csv(output_dir / "published_vs_ours_overall_comparison.csv", index=False)

print("Overall comparison table:")
display(overall_compare_df[["study", "dataset", "model_group", "model", "pearson", "pearson_delta", "pearson_delta_de", "l2"]])

# Main visual: Pearson Delta is the most directly comparable metric across all three contexts.
plot_df = overall_compare_df.copy()
plot_df["context"] = plot_df["study"] + "\n" + plot_df["dataset"].replace({"GBM radiotherapy average": "GBM RT"})
order = [
    "Ahlmann-Eltze 2025\nOverall",
    "Csendes 2025\nOverall",
    "This project\nGBM RT",
]
group_order = ["scGPT", "Simple mean", "Linear/classical"]
colors = {"scGPT": "#1f4e79", "Simple mean": "#6c757d", "Linear/classical": "#2f7d4a"}

fig, ax = plt.subplots(figsize=(10.8, 5.4))
x = np.arange(len(order))
width = 0.24
for i, group in enumerate(group_order):
    values = []
    labels = []
    for context in order:
        match = plot_df[(plot_df["context"] == context) & (plot_df["model_group"] == group)]
        if match.empty:
            values.append(np.nan)
            labels.append("")
        else:
            val = float(match["pearson_delta"].iloc[0])
            values.append(val)
            labels.append(match["model"].iloc[0])
    xpos = x + (i - 1) * width
    bars = ax.bar(xpos, values, width=width, label=group, color=colors[group], alpha=0.92)
    for bar, val, label in zip(bars, values, labels):
        if np.isfinite(val):
            ax.text(bar.get_x() + bar.get_width()/2, val + 0.015, f"{val:.3f}", ha="center", va="bottom", fontsize=8)
            ax.text(bar.get_x() + bar.get_width()/2, 0.025, label, ha="center", va="bottom", rotation=90, fontsize=7, color="white")

ax.set_xticks(x)
ax.set_xticklabels(order)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Pearson Delta")
ax.set_title("Pearson Delta: Published Benchmarks vs GBM Radiotherapy Benchmark", fontsize=13, fontweight="bold")
ax.grid(axis="y", linestyle="--", alpha=0.3)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), frameon=False)
plt.tight_layout()
plt.savefig(output_dir / "published_vs_ours_pearson_delta_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

# Secondary visual: comparable Pearson-family metrics where available.
metric_long = overall_compare_df.melt(
    id_vars=["study", "dataset", "model_group", "model"],
    value_vars=["pearson", "pearson_delta", "pearson_delta_de"],
    var_name="metric",
    value_name="value",
).dropna(subset=["value"])
metric_long["context"] = metric_long["study"] + " | " + metric_long["dataset"]
metric_long.to_csv(output_dir / "published_vs_ours_metric_long.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), sharey=True)
metric_titles = {
    "pearson": "Pearson",
    "pearson_de": "Pearson DE",
    "pearson_delta": "Pearson Delta",
    "pearson_delta_de": "Pearson DE Delta",
}
for ax, metric in zip(axes, ["pearson", "pearson_delta", "pearson_delta_de"]):
    data = metric_long[metric_long["metric"] == metric].copy()
    data["context_short"] = data["study"].map({
        "Ahlmann-Eltze 2025": "Ahlmann",
        "Csendes 2025": "Csendes",
        "This project": "Ours",
    })
    sns.barplot(data=data, x="context_short", y="value", hue="model_group", hue_order=group_order, ax=ax, palette=colors)
    ax.set_title(metric_titles[metric])
    ax.set_xlabel("")
    ax.set_ylim(0, 1.05)
    ax.grid(axis="y", linestyle="--", alpha=0.25)
    if ax is not axes[0]:
        ax.set_ylabel("")
    else:
        ax.set_ylabel("Metric value")
    if ax is axes[-1]:
        ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=False)
    else:
        ax.get_legend().remove()
fig.suptitle("Shared Pearson-Family Metrics: Published Benchmarks vs This Project", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(output_dir / "published_vs_ours_pearson_family_metrics.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved extracted-value comparison outputs:")
for filename in [
    "published_vs_ours_extracted_metric_values.csv",
    "published_vs_ours_overall_comparison.csv",
    "published_vs_ours_metric_long.csv",
    "published_vs_ours_pearson_delta_comparison.png",
    "published_vs_ours_pearson_family_metrics.png",
]:
    print(f"- {output_dir / filename}")


## Report Illustrations

These static illustrations are intended for the analysis notebook, report appendix, or presentation. They summarise the benchmark workflow, metric framework, literature positioning, and the most useful next benchmark extensions.


In [ ]:
# ==========================================================
# 10. Report illustrations
# ==========================================================
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import textwrap

illustration_dir = output_dir


def wrapped_text(text, width=28):
    return "\n".join(textwrap.wrap(text, width=width))


def add_box(ax, xy, width, height, text, face="#eef4fb", edge="#386fa4", fontsize=9, radius=0.08):
    box = FancyBboxPatch(
        xy,
        width,
        height,
        boxstyle=f"round,pad=0.03,rounding_size={radius}",
        linewidth=1.2,
        edgecolor=edge,
        facecolor=face,
    )
    ax.add_patch(box)
    ax.text(
        xy[0] + width / 2,
        xy[1] + height / 2,
        text,
        ha="center",
        va="center",
        fontsize=fontsize,
        color="#202124",
    )
    return box


def add_arrow(ax, start, end, color="#4f5b66", lw=1.5):
    arrow = FancyArrowPatch(
        start,
        end,
        arrowstyle="-|>",
        mutation_scale=14,
        linewidth=lw,
        color=color,
        shrinkA=4,
        shrinkB=4,
    )
    ax.add_patch(arrow)
    return arrow


# ---- Illustration 1: benchmark workflow ----
fig, ax = plt.subplots(figsize=(13.5, 6.2))
ax.set_xlim(0, 12)
ax.set_ylim(0, 6)
ax.axis("off")

workflow_boxes = [
    ((0.25, 3.6), 2.1, 1.05, "GBM scRNA-seq\nctrl / IR-24h / IR-1week"),
    ((2.85, 3.6), 2.0, 1.05, "Train / validation / test\nsplit"),
    ((5.35, 4.35), 2.15, 0.95, "scGPT final model\nMyriad test output"),
    ((5.35, 2.85), 2.15, 0.95, "Classical baselines\ncondition-only models"),
    ((8.05, 3.6), 2.1, 1.05, "Shared metric families\nPearson, Delta, AUPRC, distances"),
    ((10.55, 3.6), 1.25, 1.05, "Report\ninterpretation"),
]
for xy, w, h, text in workflow_boxes:
    add_box(ax, xy, w, h, text, face="#edf4fb")

add_arrow(ax, (2.35, 4.12), (2.85, 4.12))
add_arrow(ax, (4.85, 4.12), (5.35, 4.82))
add_arrow(ax, (4.85, 4.12), (5.35, 3.32))
add_arrow(ax, (7.5, 4.82), (8.05, 4.12))
add_arrow(ax, (7.5, 3.32), (8.05, 4.12))
add_arrow(ax, (10.15, 4.12), (10.55, 4.12))

ax.text(6.42, 2.25, "Comparison is project-level, not row-paired, because scGPT is loaded from final test_results.npz.",
        ha="center", va="center", fontsize=9, color="#5f6368")
ax.set_title("Benchmark Analysis Workflow", fontsize=15, fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig(illustration_dir / "illustration_benchmark_workflow.png", dpi=300, bbox_inches="tight")
plt.show()


# ---- Illustration 2: metric framework ----
metric_groups = [
    ("Global reconstruction", "Pearson\nMSE", "Does the predicted expression profile resemble the observed state overall?"),
    ("Perturbation shift", "Pearson Delta\nPearson DE Delta", "Does the model capture the direction of radiation-induced change?"),
    ("DEG recovery", "AUPRC", "Do high predicted deltas recover observed differentially changing genes?"),
    ("Distributional fidelity", "E-distance All\nLocal E-distance All\nWasserstein All\nKL All", "Does the predicted cell population resemble the observed treated distribution?"),
]
fig, ax = plt.subplots(figsize=(13.2, 5.6))
ax.set_xlim(0, 12)
ax.set_ylim(0, 5.2)
ax.axis("off")
colors = ["#e8f0fe", "#e9f7ef", "#fff4e5", "#f3e8fd"]
for i, (group, metrics, question) in enumerate(metric_groups):
    x = 0.35 + i * 2.9
    add_box(ax, (x, 2.55), 2.45, 1.15, f"{group}\n{metrics}", face=colors[i], edge="#5f6368", fontsize=8.5)
    add_box(ax, (x, 0.65), 2.45, 1.35, wrapped_text(question, 24), face="#ffffff", edge="#d0d4d9", fontsize=8.2, radius=0.05)
    add_arrow(ax, (x + 1.22, 2.55), (x + 1.22, 2.0), color="#77808a", lw=1.2)
ax.text(6, 4.45, "Report-facing metric set after removing redundant Spearman and Common-DEGs", ha="center", fontsize=10, color="#5f6368")
ax.set_title("Metric Framework for the GBM Radiotherapy Benchmark", fontsize=15, fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig(illustration_dir / "illustration_metric_framework.png", dpi=300, bbox_inches="tight")
plt.show()


# ---- Illustration 3: literature positioning map ----
fig, ax = plt.subplots(figsize=(12.8, 6.4))
ax.set_xlim(0, 10)
ax.set_ylim(0, 7)
ax.axis("off")

# Axes as conceptual gradients.
ax.annotate("", xy=(9.3, 1), xytext=(1, 1), arrowprops=dict(arrowstyle="-|>", color="#4f5b66", lw=1.4))
ax.annotate("", xy=(1, 6.3), xytext=(1, 1), arrowprops=dict(arrowstyle="-|>", color="#4f5b66", lw=1.4))
ax.text(5.15, 0.35, "Perturbation context: standard public screen  →  dataset-specific therapeutic stress", ha="center", fontsize=9)
ax.text(0.1, 3.7, "Evaluation emphasis:\nsimple metrics →\nbiology-aware / distributional", ha="left", va="center", fontsize=9)

points = [
    (2.2, 2.0, "Ahlmann\n2025", "simple baselines strong"),
    (2.9, 2.6, "Csendes\n2025", "train-mean / RF strong"),
    (3.9, 4.2, "Bendidi\n2024", "PCA / scVI competitive"),
    (4.8, 5.2, "PerturBench\n2024/25", "split + metric choice matter"),
    (8.0, 5.3, "This project", "GBM radiotherapy\nscGPT vs baselines"),
]
for x, y, label, note in points:
    face = "#d7ecff" if label == "This project" else "#f2f4f7"
    edge = "#1f4e79" if label == "This project" else "#8a939e"
    add_box(ax, (x - 0.75, y - 0.42), 1.5, 0.84, label, face=face, edge=edge, fontsize=8.5)
    ax.text(x, y - 0.72, note, ha="center", va="top", fontsize=7.6, color="#5f6368")

ax.text(7.95, 6.35, "Compare interpretation patterns, not direct leaderboard scores", ha="center", fontsize=9, color="#5f6368")
ax.set_title("How This Benchmark Sits Relative to Published Perturbation Benchmarks", fontsize=15, fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig(illustration_dir / "illustration_literature_positioning.png", dpi=300, bbox_inches="tight")
plt.show()


# ---- Illustration 4: next extensions roadmap ----
fig, ax = plt.subplots(figsize=(13.2, 5.8))
ax.set_xlim(0, 12)
ax.set_ylim(0, 5.6)
ax.axis("off")

left_items = [
    ("RMSE / L2 error", "Aligns with Ahlmann-style error metrics"),
    ("Variance-compression ratio", "Detects under-amplified radiation response"),
    ("Pathway-level delta agreement", "Connects prediction to GBM radiobiology"),
]
right_items = [
    ("PCA-ridge baseline", "Strong interpretable classical comparator"),
    ("Control-matched ridge", "Closer counterfactual baseline than condition lookup"),
    ("Biology/QC-feature RF or boosting", "Tests Csendes-style informed ML baseline"),
]

add_box(ax, (0.55, 4.35), 4.9, 0.7, "Suggested Additional Metrics", face="#e8f0fe", edge="#386fa4", fontsize=10)
add_box(ax, (6.55, 4.35), 4.9, 0.7, "Suggested Additional Models", face="#e9f7ef", edge="#2f7d4a", fontsize=10)

for i, (title, subtitle) in enumerate(left_items):
    y = 3.3 - i * 1.1
    add_box(ax, (0.75, y), 4.5, 0.75, f"{title}\n{subtitle}", face="#ffffff", edge="#d0d4d9", fontsize=8.5, radius=0.05)
for i, (title, subtitle) in enumerate(right_items):
    y = 3.3 - i * 1.1
    add_box(ax, (6.75, y), 4.5, 0.75, f"{title}\n{subtitle}", face="#ffffff", edge="#d0d4d9", fontsize=8.5, radius=0.05)

ax.text(6, 0.35, "Highest priority: add RMSE/L2 and PCA-ridge or control-matched ridge if time allows.", ha="center", fontsize=9.2, color="#5f6368")
ax.set_title("Practical Extensions for a Stronger Benchmark Section", fontsize=15, fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig(illustration_dir / "illustration_next_extensions.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved report illustrations:")
for filename in [
    "illustration_benchmark_workflow.png",
    "illustration_metric_framework.png",
    "illustration_literature_positioning.png",
    "illustration_next_extensions.png",
]:
    print(f"- {illustration_dir / filename}")


In [ ]:
plt.hist(X_test.mean(axis=0))